# Notebook 17 — Robustness, Interpretation and Final Outputs

This notebook completes Phase 6 for the **Version 2 exact-t−1 lagged-bank main analysis**. It performs an exploratory crisis-era exercise, a controlled archived Version 1 sensitivity comparison, a descriptive recent-period uninsured-deposit analysis, and the final dissertation tables and figures.

The crisis exercise uses eligible Version 2 reference rows from 2001Q2–2006Q4 for training, excludes calendar 2007, and evaluates 2008Q1–2012Q4. All crisis preprocessing and class weights are refitted on the pre-crisis training rows only. Hyperparameters are fixed in this notebook and do not come from Notebook 15. Crisis findings are exploratory because pre-crisis failures are sparse and the predictor-selection data overlap the evaluation period.

The final-test sensitivity comparison never refits Version 1. It reports archived Version 1 on its original denominator, archived Version 1 restricted to Version 2-eligible keys, and Version 2 on those same keys. Version 2 remains the main analysis regardless of comparative performance. Differences are non-causal and reflect both predictor timing and the longer effective horizon from the predictor date; they are not a pure timing effect.

In [ ]:
# Resolve Version 2, archived Version 1 and fixed Phase 6 contracts

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import math
import os
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import sklearn
import xgboost as xgb
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


COLAB_MYDRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None
if not COLAB_MYDRIVE.is_dir() and drive is not None:
    drive.mount("/content/drive", force_remount=False)

DEFAULT_PROJECT_ROOT = Path(os.environ.get(
    "DISSERTATION_PROJECT_ROOT", "/content/drive/MyDrive/Data Final"
)).expanduser()
explicit_config = os.environ.get("V2_CONFIG_PATH")
CONFIG_PATH = Path(explicit_config).expanduser() if explicit_config else (
    DEFAULT_PROJECT_ROOT / "Version 2 Main Analysis" / "config.py"
)
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Shared Version 2 config not found: {CONFIG_PATH}")

config_spec = importlib.util.spec_from_file_location("v2_config", CONFIG_PATH)
config = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(config)
ANALYSIS_VERSION = str(config.ANALYSIS_VERSION)
TIMING_POLICY = dict(config.VERSION_2_MAIN_TIMING_POLICY)
TARGET_HORIZON_DAYS = int(config.TARGET_HORIZON_DAYS)
RANDOM_SEED = int(config.RANDOM_SEED)
V2_OUTPUT_ROOT = Path(config.V2_OUTPUT_ROOT).expanduser()
PROJECT_ROOT = Path(getattr(config, "PROJECT_ROOT", CONFIG_PATH.parent.parent)).expanduser()
V2_ROOT = Path(getattr(config, "V2_ROOT", CONFIG_PATH.parent)).expanduser()
HISTORY_POLICY = dict(config.HISTORY_POLICY)

TARGET = "failed_1yr"
PANEL_KEYS = ["report_date", "cert", "idrssd"]
JOIN_KEYS = ["report_date", "idrssd"]
HISTORY_INDICATOR = str(HISTORY_POLICY["indicator_model_name"])
MODEL_IDS = [
    "logistic_regression__bank_only", "logistic_regression__bank_plus_macro",
    "random_forest__bank_only", "random_forest__bank_plus_macro",
    "xgboost__bank_only", "xgboost__bank_plus_macro",
]
MODEL_FAMILIES = ["logistic_regression", "random_forest", "xgboost"]
SPECIFICATIONS = ["bank_only", "bank_plus_macro"]
TOP_RISK_PERCENTAGES = (0.01, 0.05, 0.10, 0.20)
CRISIS_TRAIN_START, CRISIS_TRAIN_END = pd.Timestamp("2001-06-30"), pd.Timestamp("2006-12-31")
CRISIS_PURGE_START, CRISIS_PURGE_END = pd.Timestamp("2007-03-31"), pd.Timestamp("2007-12-31")
CRISIS_EVAL_START, CRISIS_EVAL_END = pd.Timestamp("2008-03-31"), pd.Timestamp("2012-12-31")
WINSOR_LOWER_QUANTILE, WINSOR_UPPER_QUANTILE = 0.01, 0.99
BOOTSTRAP_SEED = 20260816
BOOTSTRAP_REPLICATES = 1000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
BASELINE_ID = "low_equity_to_assets_lag1_baseline"
FDIC_FINANCIALS_URL = "https://api.fdic.gov/banks/financials"

# Fixed here, before crisis outcomes; no Notebook 15-selected hyperparameter is read.
CRISIS_SETTINGS = {
    "logistic_regression": {
        "penalty": "l2", "C": 1.0, "solver": "lbfgs", "max_iter": 1000,
        "positive_weight_multiplier": 1.0,
    },
    "random_forest": {
        "n_estimators": 300, "max_depth": None, "min_samples_leaf": 20,
        "max_features": "sqrt", "max_samples": 0.5,
        "positive_weight_multiplier": 1.0,
    },
    "xgboost": {
        "n_estimators": 300, "max_depth": 3, "learning_rate": 0.05,
        "min_child_weight": 5.0, "subsample": 0.8, "colsample_bytree": 0.8,
        "reg_lambda": 1.0, "reg_alpha": 0.0,
        "positive_weight_multiplier": 1.0,
    },
}


def deduplicate_paths(paths):
    result = []
    for value in paths:
        path = Path(value).expanduser()
        if path not in result:
            result.append(path)
    return result


def bundle_complete(root, relative_paths):
    return all((root / relative).is_file() for relative in relative_paths)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def passed_values(series):
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "pass"})


def quarter_end(label):
    return pd.Period(str(label), freq="Q").end_time.normalize()


def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return value


def strict_json_dump(payload, path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "w", encoding="utf-8") as handle:
        json.dump(to_builtin(payload), handle, indent=2, sort_keys=True, allow_nan=False)
    temporary.replace(path)


data_bundle = [
    Path("09_Final_Modelling_Panel") / "v2_final_modelling_panel_2001Q2_2025Q2.parquet",
    Path("09_Final_Modelling_Panel") / "v2_positive_predictor_availability_margin_audit.parquet",
    Path("09_Final_Modelling_Panel") / "v2_cert_mismatch_allowlist.csv",
    Path("09_Final_Modelling_Panel") / "cell_08_final_validation_gate.csv",
    Path("10_Chronological_Validation_Design") / "v2_chronological_row_membership.parquet",
    Path("10_Chronological_Validation_Design") / "cell_07_final_validation_gate.csv",
]
lock_bundle = [
    Path("15_Validation_Comparison_and_Model_Lock") / "v2_model_lock_manifest.json",
    Path("15_Validation_Comparison_and_Model_Lock") / "cell_07_final_validation_gate.csv",
    Path("16_Final_Out_of_Time_Test") / "v2_notebook_16_to_17_handoff.json",
    Path("16_Final_Out_of_Time_Test") / "cell_09_final_validation_gate.csv",
]
candidate_roots = deduplicate_paths([
    V2_OUTPUT_ROOT / ANALYSIS_VERSION, V2_OUTPUT_ROOT,
    V2_ROOT / "Notebook Outputs" / ANALYSIS_VERSION, V2_ROOT / "Notebook Outputs",
    PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION, PROJECT_ROOT / "Notebook Outputs",
    DEFAULT_PROJECT_ROOT / "Notebook Outputs" / ANALYSIS_VERSION,
])
data_roots = [root for root in candidate_roots if bundle_complete(root, data_bundle)]
lock_roots = [root for root in candidate_roots if bundle_complete(root, lock_bundle)]
if not data_roots or not lock_roots:
    raise FileNotFoundError("A complete Version 2 Notebook 09/15/16 handoff bundle was not found.")

DATA_OUTPUT_ROOT, LOCK_OUTPUT_ROOT = data_roots[0], lock_roots[0]
NB09_OUTPUT_DIR = DATA_OUTPUT_ROOT / "09_Final_Modelling_Panel"
NB10_OUTPUT_DIR = DATA_OUTPUT_ROOT / "10_Chronological_Validation_Design"
NB15_OUTPUT_DIR = LOCK_OUTPUT_ROOT / "15_Validation_Comparison_and_Model_Lock"
NB16_OUTPUT_DIR = LOCK_OUTPUT_ROOT / "16_Final_Out_of_Time_Test"
NB17_OUTPUT_DIR = LOCK_OUTPUT_ROOT / "17_Robustness_Interpretation_and_Final_Outputs"
CRISIS_MODEL_DIR = NB17_OUTPUT_DIR / "crisis_models"
CRISIS_PREDICTION_DIR = NB17_OUTPUT_DIR / "crisis_predictions"
CRISIS_PREPROCESSING_DIR = NB17_OUTPUT_DIR / "crisis_preprocessing"
CRISIS_CHECKPOINT_DIR = NB17_OUTPUT_DIR / "crisis_checkpoints"
FIGURE_DIR = NB17_OUTPUT_DIR / "figures"
for directory in [NB17_OUTPUT_DIR, CRISIS_MODEL_DIR, CRISIS_PREDICTION_DIR,
                  CRISIS_PREPROCESSING_DIR, CRISIS_CHECKPOINT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

FINAL_PANEL_PATH = NB09_OUTPUT_DIR / "v2_final_modelling_panel_2001Q2_2025Q2.parquet"
HORIZON_AUDIT_PATH = NB09_OUTPUT_DIR / "v2_positive_predictor_availability_margin_audit.parquet"
CERT_ALLOWLIST_PATH = NB09_OUTPUT_DIR / "v2_cert_mismatch_allowlist.csv"
NB09_GATE_PATH = NB09_OUTPUT_DIR / "cell_08_final_validation_gate.csv"
MEMBERSHIP_PATH = NB10_OUTPUT_DIR / "v2_chronological_row_membership.parquet"
NB10_GATE_PATH = NB10_OUTPUT_DIR / "cell_07_final_validation_gate.csv"
MODEL_LOCK_PATH = NB15_OUTPUT_DIR / "v2_model_lock_manifest.json"
NB15_GATE_PATH = NB15_OUTPUT_DIR / "cell_07_final_validation_gate.csv"
NB16_HANDOFF_PATH = NB16_OUTPUT_DIR / "v2_notebook_16_to_17_handoff.json"
NB16_GATE_PATH = NB16_OUTPUT_DIR / "cell_09_final_validation_gate.csv"

with open(NB16_HANDOFF_PATH, "r", encoding="utf-8") as handle:
    nb16_handoff = json.load(handle)
V2_PREDICTION_DIR = Path(nb16_handoff["principal_outputs"]["predictions_directory"])
V2_PREDICTION_PATHS = {
    model_id: V2_PREDICTION_DIR / f"{model_id}__v2_final_test_predictions.parquet"
    for model_id in MODEL_IDS
}
NB16_FEATURE_EFFECTS_PATH = NB16_OUTPUT_DIR / "v2_final_model_feature_effects.csv"
NB16_CALIBRATION_PATH = NB16_OUTPUT_DIR / "v2_calibration_diagnostic_summary.csv"
NB16_HORIZON_PATH = Path(nb16_handoff["principal_outputs"]["horizon"])

v1_relative_predictions = [
    Path("final_test_predictions") / f"{model_id}__final_test_predictions.parquet"
    for model_id in MODEL_IDS
]
explicit_v1 = os.environ.get("ARCHIVED_V1_NB16_OUTPUT_DIR", "").strip()
v1_candidates = deduplicate_paths([
    *( [Path(explicit_v1)] if explicit_v1 else [] ),
    PROJECT_ROOT / "Notebook Outputs" / "16_Final_Out_of_Time_Test",
    DEFAULT_PROJECT_ROOT / "Notebook Outputs" / "16_Final_Out_of_Time_Test",
    DEFAULT_PROJECT_ROOT / "Version 1 Main Analysis" / "Notebook Outputs" / "16_Final_Out_of_Time_Test",
])
v1_roots = [root for root in v1_candidates if bundle_complete(root, v1_relative_predictions)]
if not v1_roots:
    raise FileNotFoundError(
        "Archived Version 1 prediction bundle not found. Set ARCHIVED_V1_NB16_OUTPUT_DIR."
    )
V1_NB16_OUTPUT_DIR = v1_roots[0]
V1_PREDICTION_PATHS = {
    model_id: V1_NB16_OUTPUT_DIR / "final_test_predictions" / f"{model_id}__final_test_predictions.parquet"
    for model_id in MODEL_IDS
}
explicit_v1_lock = os.environ.get("ARCHIVED_V1_MODEL_LOCK_PATH", "").strip()
V1_MODEL_LOCK_PATH = Path(explicit_v1_lock).expanduser() if explicit_v1_lock else (
    V1_NB16_OUTPUT_DIR.parent / "15_Validation_Comparison_and_Model_Lock" / "locked_model_specifications.json"
)
if not V1_MODEL_LOCK_PATH.is_file():
    raise FileNotFoundError(
        "Archived Version 1 model lock not found. Set ARCHIVED_V1_MODEL_LOCK_PATH."
    )
v1_crisis_relative = [
    Path("crisis_discrimination_with_clustered_ci.csv"),
    Path("crisis_capacity_with_clustered_ci.csv"),
]
explicit_v1_nb17 = os.environ.get("ARCHIVED_V1_NB17_OUTPUT_DIR", "").strip()
v1_nb17_candidates = deduplicate_paths([
    *( [Path(explicit_v1_nb17)] if explicit_v1_nb17 else [] ),
    V1_NB16_OUTPUT_DIR.parent / "17_Robustness_Interpretation_and_Final_Outputs",
])
v1_nb17_roots = [root for root in v1_nb17_candidates if bundle_complete(root, v1_crisis_relative)]
if not v1_nb17_roots:
    raise FileNotFoundError(
        "Archived Version 1 crisis result bundle not found. Set ARCHIVED_V1_NB17_OUTPUT_DIR."
    )
V1_NB17_OUTPUT_DIR = v1_nb17_roots[0]
V1_CRISIS_DISCRIMINATION_PATH = V1_NB17_OUTPUT_DIR / "crisis_discrimination_with_clustered_ci.csv"
V1_CRISIS_CAPACITY_PATH = V1_NB17_OUTPUT_DIR / "crisis_capacity_with_clustered_ci.csv"

required_inputs = [
    CONFIG_PATH, FINAL_PANEL_PATH, HORIZON_AUDIT_PATH, CERT_ALLOWLIST_PATH,
    NB09_GATE_PATH, MEMBERSHIP_PATH, NB10_GATE_PATH, MODEL_LOCK_PATH,
    NB15_GATE_PATH, NB16_HANDOFF_PATH, NB16_GATE_PATH, NB16_FEATURE_EFFECTS_PATH,
    NB16_CALIBRATION_PATH, NB16_HORIZON_PATH, V1_MODEL_LOCK_PATH,
    V1_CRISIS_DISCRIMINATION_PATH, V1_CRISIS_CAPACITY_PATH,
    *V2_PREDICTION_PATHS.values(), *V1_PREDICTION_PATHS.values(),
]
input_hashes_before = {str(path): sha256_file(path) for path in required_inputs}

phase_6_contract = pd.DataFrame({
    "item": [
        "crisis training", "crisis purge", "crisis evaluation", "timing",
        "crisis settings", "crisis preprocessing", "crisis status",
        "V1 full sample", "like-for-like comparison", "main analysis",
        "bootstrap", "capacity", "comparison interpretation", "uninsured deposits",
    ],
    "locked_rule": [
        "eligible Version 2 reference rows 2001Q2-2006Q4; counts recalculated",
        "calendar 2007 excluded and not loaded",
        "eligible Version 2 reference rows 2008Q1-2012Q4",
        "exact t-1 bank predictors, t-1 macro predictors, target (t,t+365]",
        "fixed in Notebook 17; independent of Notebook 15 tuning",
        "medians, 1st/99th bounds, LR scaler and class weights fitted on crisis training only",
        "exploratory robustness; sparse pre-crisis failures and predictor-selection overlap disclosed",
        "archived original denominator; secondary and explicitly non-like-for-like",
        "V1 predictions joined to V2-eligible report_date+IDRSSD keys; identical quarterly group sizes",
        "Version 2 remains the headline analysis regardless of PR-AUC direction",
        "audited CERT-component identity; multinomial multiplicities; seed 20260816; 1,000 percentile replicates",
        "quarterly top 1/5/10/20%; failed-bank capture primary; positive bank-quarter capture secondary",
        "non-causal; timing plus longer effective horizon; feature-set differences disclosed",
        "DEPUNINS/DEP for recent eligible banks and failures; descriptive only",
    ],
})
cell_01_checks = {
    "The Version 2 timing contract remains exact t-1 with a 365-day target":
        ANALYSIS_VERSION == "v2_bank_lag1_main"
        and TIMING_POLICY.get("bank_predictor_source") == "exact_t_minus_1"
        and TIMING_POLICY.get("macro_predictor_source") == "t_minus_1"
        and TARGET_HORIZON_DAYS == 365,
    "Crisis dates are 2001Q2-2006Q4, 2007 purge and 2008Q1-2012Q4 evaluation":
        (CRISIS_TRAIN_START, CRISIS_TRAIN_END, CRISIS_PURGE_START, CRISIS_PURGE_END,
         CRISIS_EVAL_START, CRISIS_EVAL_END) == (
            pd.Timestamp("2001-06-30"), pd.Timestamp("2006-12-31"),
            pd.Timestamp("2007-03-31"), pd.Timestamp("2007-12-31"),
            pd.Timestamp("2008-03-31"), pd.Timestamp("2012-12-31")),
    "All fixed crisis settings use a neutral class-weight multiplier":
        set(CRISIS_SETTINGS) == set(MODEL_FAMILIES)
        and all(v["positive_weight_multiplier"] == 1.0 for v in CRISIS_SETTINGS.values()),
    "The comparison bootstrap is identical to Version 1":
        BOOTSTRAP_SEED == 20260816 and BOOTSTRAP_REPLICATES == 1000,
    "All required Version 2 and archived Version 1 inputs exist":
        all(path.is_file() for path in required_inputs),
    "Notebook 16 remains locked without test-based reselection":
        nb16_handoff["test_based_reselection_performed"] is False,
}
notebook_17_cell_01_gate = pd.DataFrame({
    "check": cell_01_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_01_checks.values()],
    "passed": cell_01_checks.values(),
})
phase_6_contract.to_csv(NB17_OUTPUT_DIR / "cell_01_phase_6_contract.csv", index=False)
pd.DataFrame([{"model_family": k, **v} for k, v in CRISIS_SETTINGS.items()]).to_csv(
    NB17_OUTPUT_DIR / "cell_01_fixed_crisis_settings.csv", index=False
)
notebook_17_cell_01_gate.to_csv(NB17_OUTPUT_DIR / "cell_01_setup_gate.csv", index=False)
print("Notebook 17 Cell 1 validation gate:")
display(notebook_17_cell_01_gate)
print("\nPhase 6 contract:")
with pd.option_context("display.max_colwidth", None):
    display(phase_6_contract)
failed = [name for name, passed in cell_01_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 1 blocked: " + "; ".join(failed))
print("\nCell 1 completed successfully.")
print("Version 2 output root:", LOCK_OUTPUT_ROOT)
print("Archived Version 1 Notebook 16 root:", V1_NB16_OUTPUT_DIR)
print("Notebook 17 output directory:", NB17_OUTPUT_DIR)

In [ ]:
# Audit handoffs, isolate predictor lists, and disclose feature-set differences

if not notebook_17_cell_01_gate["passed"].all():
    raise AssertionError("Cell 2 blocked because Cell 1 did not pass.")

nb15_gate = pd.read_csv(NB15_GATE_PATH)
nb16_gate = pd.read_csv(NB16_GATE_PATH)
with open(MODEL_LOCK_PATH, "r", encoding="utf-8") as handle:
    v2_lock = json.load(handle)
with open(V1_MODEL_LOCK_PATH, "r", encoding="utf-8") as handle:
    v1_lock = json.load(handle)
final_v2_feature_effects = pd.read_csv(NB16_FEATURE_EFFECTS_PATH)
final_v2_calibration = pd.read_csv(NB16_CALIBRATION_PATH)
final_v2_horizon = pd.read_csv(NB16_HORIZON_PATH)
archived_v1_crisis_discrimination = pd.read_csv(V1_CRISIS_DISCRIMINATION_PATH)
archived_v1_crisis_capacity = pd.read_csv(V1_CRISIS_CAPACITY_PATH)

leader_id = nb16_handoff["validation_selected_leader"]


def normalise_lock_records(lock, version_label):
    if "locked_models" in lock:
        raw_records = lock["locked_models"]
    elif "models" in lock:
        raw_records = lock["models"]
    else:
        raise KeyError(
            f"{version_label} model lock contains neither 'locked_models' nor 'models'. "
            f"Available top-level keys: {sorted(lock)}"
        )
    records = {}
    for raw_record in raw_records:
        record = dict(raw_record)
        predictors = record.get("predictors_in_order", record.get("predictors"))
        if predictors is None:
            raise KeyError(
                f"{version_label} record {record.get('model_id', '<unknown>')} contains neither "
                "'predictors_in_order' nor 'predictors'."
            )
        record["predictors"] = list(predictors)
        required = {"model_id", "model_family", "specification"}
        missing = sorted(required - set(record))
        if missing:
            raise KeyError(
                f"{version_label} record is missing required fields: {', '.join(missing)}"
            )
        if record["model_id"] in records:
            raise AssertionError(f"Duplicate {version_label} model_id: {record['model_id']}")
        records[record["model_id"]] = record
    return records


v2_records = normalise_lock_records(v2_lock, "Version 2")
v1_records = normalise_lock_records(v1_lock, "archived Version 1")
if set(v2_records) != set(MODEL_IDS) or set(v1_records) != set(MODEL_IDS):
    raise AssertionError("Version 1 and Version 2 locks must each contain the same six model IDs.")

predictor_sets = {}
for specification in SPECIFICATIONS:
    values = [
        list(record["predictors"])
        for record in v2_records.values()
        if record["specification"] == specification
    ]
    if not values or any(value != values[0] for value in values[1:]):
        raise AssertionError(f"Version 2 predictor lists disagree within {specification}.")
    predictor_sets[specification] = values[0]
all_predictors = list(dict.fromkeys(predictor_sets["bank_plus_macro"]))

v2_bank_predictors = set(predictor_sets["bank_only"])


def normalise_v2_feature(feature):
    # Only bank predictors receive the new lag1 suffix. Macro predictors were already lagged in V1.
    if feature in v2_bank_predictors and feature.endswith("_lag1"):
        return feature[:-5]
    return feature


feature_rows = []
feature_summary_rows = []
for specification in SPECIFICATIONS:
    v1_lists = [
        list(record["predictors"])
        for record in v1_records.values()
        if record["specification"] == specification
    ]
    if not v1_lists or any(value != v1_lists[0] for value in v1_lists[1:]):
        raise AssertionError(f"Archived Version 1 predictor lists disagree within {specification}.")
    v1_features = v1_lists[0]
    v2_features = predictor_sets[specification]
    v2_normalised = [normalise_v2_feature(value) for value in v2_features]
    union = sorted(set(v1_features) | set(v2_normalised))
    for feature in union:
        feature_rows.append({
            "specification": specification,
            "normalised_feature": feature,
            "in_archived_version_1": feature in v1_features,
            "in_version_2_after_timing_name_normalisation": feature in v2_normalised,
            "status": (
                "same feature"
                if feature in v1_features and feature in v2_normalised
                else "Version 1 only" if feature in v1_features else "Version 2 only"
            ),
        })
    feature_summary_rows.append({
        "specification": specification,
        "version_1_predictor_count": len(v1_features),
        "version_2_predictor_count": len(v2_features),
        "normalised_feature_sets_identical": set(v1_features) == set(v2_normalised),
        "version_1_only": " | ".join(sorted(set(v1_features) - set(v2_normalised))),
        "version_2_only": " | ".join(sorted(set(v2_normalised) - set(v1_features))),
        "disclosure": "No feature-set difference after timing-name normalisation"
            if set(v1_features) == set(v2_normalised)
            else "Feature-set difference is a comparison confounder",
    })

feature_reconciliation = pd.DataFrame(feature_rows)
feature_set_summary = pd.DataFrame(feature_summary_rows)
predictor_manifest = pd.DataFrame([
    {
        "specification": specification,
        "predictor_count": len(values),
        "predictors": " | ".join(values),
        "source": "Version 2 locked predictor list only; no Notebook 15 hyperparameter used",
    }
    for specification, values in predictor_sets.items()
])
cell_02_checks = {
    "Notebook 15 and Notebook 16 gates re-read fully passed":
        passed_values(nb15_gate["passed"]).all() and passed_values(nb16_gate["passed"]).all(),
    "The validation-selected leader remains Random Forest bank plus macro":
        leader_id == "random_forest__bank_plus_macro",
    "Version 2 has exactly 19 bank-only and 23 bank-plus-macro predictors":
        len(predictor_sets["bank_only"]) == 19 and len(predictor_sets["bank_plus_macro"]) == 23,
    "Both Version 2 specifications retain the lagged history indicator":
        all(HISTORY_INDICATOR in predictor_sets[s] for s in SPECIFICATIONS),
    "Feature reconciliation explicitly covers both specifications":
        set(feature_set_summary["specification"]) == set(SPECIFICATIONS)
        and feature_reconciliation.groupby("specification").size().gt(0).all(),
    "Notebook 16 interpretation, calibration and realised-horizon outputs are available":
        set(final_v2_feature_effects["model_id"]) == set(MODEL_IDS)
        and set(final_v2_calibration["model_id"]) == set(MODEL_IDS)
        and len(final_v2_horizon) > 0,
    "Archived Version 1 unlagged crisis results are retained as secondary evidence":
        len(archived_v1_crisis_discrimination) == 7
        and len(archived_v1_crisis_capacity) == 28,
    "No archived or Version 2 test metric enters crisis settings":
        all("selected_hyperparameters" not in settings for settings in CRISIS_SETTINGS.values()),
}
notebook_17_cell_02_gate = pd.DataFrame({
    "check": cell_02_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_02_checks.values()],
    "passed": cell_02_checks.values(),
})
predictor_manifest.to_csv(NB17_OUTPUT_DIR / "cell_02_crisis_predictor_manifest.csv", index=False)
feature_reconciliation.to_csv(NB17_OUTPUT_DIR / "v1_v2_feature_reconciliation.csv", index=False)
feature_set_summary.to_csv(NB17_OUTPUT_DIR / "v1_v2_feature_set_summary.csv", index=False)
notebook_17_cell_02_gate.to_csv(NB17_OUTPUT_DIR / "cell_02_handoff_gate.csv", index=False)
print("Notebook 17 Cell 2 validation gate:")
display(notebook_17_cell_02_gate)
print("\nVersion 1-Version 2 feature-set disclosure:")
display(feature_set_summary)
failed = [name for name, passed in cell_02_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 2 blocked: " + "; ".join(failed))
print("\nCell 2 completed successfully. Only predictor lists were isolated; crisis settings remain independent.")

In [ ]:
# Recalculate eligible crisis samples after exact-predecessor exclusions

if not notebook_17_cell_02_gate["passed"].all():
    raise AssertionError("Cell 3 blocked because Cell 2 did not pass.")


def audited_cert_components(allowlist):
    parent = {}

    def find(value):
        parent.setdefault(value, value)
        if parent[value] != value:
            parent[value] = find(parent[value])
        return parent[value]

    def union(left, right):
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            smaller, larger = sorted((left_root, right_root))
            parent[larger] = smaller

    for row in allowlist.itertuples(index=False):
        union(int(row.cert_source), int(row.cert_reference))
    components = {}
    for value in list(parent):
        components.setdefault(find(value), []).append(value)
    identity_map, audit_rows = {}, []
    for members in components.values():
        canonical = min(members)
        for value in members:
            identity_map[value] = canonical
            audit_rows.append({
                "original_cert": value, "bootstrap_identity": canonical,
                "component_size": len(members),
            })
    return identity_map, pd.DataFrame(audit_rows)


cert_allowlist = pd.read_csv(CERT_ALLOWLIST_PATH)
cert_identity_map, cert_identity_audit = audited_cert_components(cert_allowlist)
nb09_gate = pd.read_csv(NB09_GATE_PATH)
nb10_gate = pd.read_csv(NB10_GATE_PATH)
panel_columns = list(dict.fromkeys([*PANEL_KEYS, *all_predictors, TARGET]))
crisis_train_panel = pd.read_parquet(
    FINAL_PANEL_PATH, columns=panel_columns,
    filters=[("report_date", ">=", CRISIS_TRAIN_START), ("report_date", "<=", CRISIS_TRAIN_END)],
)
crisis_eval_panel = pd.read_parquet(
    FINAL_PANEL_PATH, columns=panel_columns,
    filters=[("report_date", ">=", CRISIS_EVAL_START), ("report_date", "<=", CRISIS_EVAL_END)],
)
for frame in [crisis_train_panel, crisis_eval_panel]:
    frame["report_date"] = pd.to_datetime(frame["report_date"], errors="raise").dt.normalize()
    frame["cert"] = pd.to_numeric(frame["cert"], errors="raise").astype("int64")
    frame["idrssd"] = pd.to_numeric(frame["idrssd"], errors="raise").astype("int64")
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors="raise").astype("int8")

# panel_row_id is owned by Notebook 10 membership, not duplicated in the modelling Parquet.
membership = pd.read_parquet(
    MEMBERSHIP_PATH, columns=["panel_row_id", *PANEL_KEYS, TARGET]
)
membership["report_date"] = pd.to_datetime(
    membership["report_date"], errors="raise"
).dt.normalize()
membership["cert"] = pd.to_numeric(membership["cert"], errors="raise").astype("int64")
membership["idrssd"] = pd.to_numeric(membership["idrssd"], errors="raise").astype("int64")
membership[TARGET] = pd.to_numeric(membership[TARGET], errors="raise").astype("int8")
if membership["panel_row_id"].duplicated().any() or membership.duplicated(PANEL_KEYS).any():
    raise AssertionError("Notebook 10 membership keys are not unique.")


def attach_panel_row_id(panel_frame):
    scoped_membership = membership.loc[
        membership["report_date"].isin(panel_frame["report_date"].unique()),
        ["panel_row_id", *PANEL_KEYS, TARGET],
    ]
    merged = panel_frame.merge(
        scoped_membership, on=PANEL_KEYS, how="inner",
        suffixes=("_panel", "_membership"), validate="one_to_one",
    )
    if len(merged) != len(panel_frame):
        raise AssertionError("A crisis panel row is missing from Notebook 10 membership.")
    if not merged[f"{TARGET}_panel"].eq(merged[f"{TARGET}_membership"]).all():
        raise AssertionError("Notebook 9 and Notebook 10 targets disagree in the crisis sample.")
    merged[TARGET] = merged.pop(f"{TARGET}_panel").astype("int8")
    merged.drop(columns=[f"{TARGET}_membership"], inplace=True)
    merged.sort_values("panel_row_id", kind="mergesort", inplace=True)
    merged.reset_index(drop=True, inplace=True)
    return merged


crisis_train_data = attach_panel_row_id(crisis_train_panel)
crisis_eval_data = attach_panel_row_id(crisis_eval_panel)
for frame in [crisis_train_data, crisis_eval_data]:
    frame["bootstrap_identity"] = frame["cert"].map(cert_identity_map).fillna(frame["cert"]).astype("int64")

# The positive-row availability audit carries source_report_date. Notebook 9's passed
# final gate covers the exact-predecessor match for the full panel; this scoped audit
# independently verifies it for every crisis positive bank-quarter.
crisis_horizon_audit = pd.read_parquet(HORIZON_AUDIT_PATH)
crisis_horizon_audit["report_date"] = pd.to_datetime(
    crisis_horizon_audit["report_date"], errors="raise"
).dt.normalize()
crisis_horizon_audit["source_report_date"] = pd.to_datetime(
    crisis_horizon_audit["source_report_date"], errors="raise"
).dt.normalize()
crisis_horizon_audit = crisis_horizon_audit.loc[
    crisis_horizon_audit["report_date"].between(CRISIS_TRAIN_START, CRISIS_TRAIN_END)
    | crisis_horizon_audit["report_date"].between(CRISIS_EVAL_START, CRISIS_EVAL_END)
].copy()
crisis_horizon_audit["exact_predecessor"] = [
    pd.Period(source, freq="Q") == pd.Period(reference, freq="Q") - 1
    for source, reference in zip(
        crisis_horizon_audit["source_report_date"], crisis_horizon_audit["report_date"]
    )
]

training_positives = int(crisis_train_data[TARGET].sum())
training_negatives = int(len(crisis_train_data) - training_positives)
training_failed_banks = int(crisis_train_data.loc[
    crisis_train_data[TARGET].eq(1), "bootstrap_identity"
].nunique())
if training_positives <= 0:
    raise AssertionError("Crisis training contains no positive bank-quarter.")
CRISIS_IMBALANCE_RATIO = training_negatives / training_positives
eval_positives = int(crisis_eval_data[TARGET].sum())
eval_failed_banks = int(crisis_eval_data.loc[
    crisis_eval_data[TARGET].eq(1), "bootstrap_identity"
].nunique())

crisis_sample_summary = pd.DataFrame({
    "sample": ["pre_crisis_training", "purge_not_loaded", "crisis_evaluation"],
    "reference_period": ["2001Q2-2006Q4", "2007Q1-2007Q4", "2008Q1-2012Q4"],
    "observations": [len(crisis_train_data), np.nan, len(crisis_eval_data)],
    "positive_bank_quarters": [training_positives, np.nan, eval_positives],
    "distinct_failed_banks_audited_identity": [training_failed_banks, np.nan, eval_failed_banks],
    "quarters": [crisis_train_data["report_date"].nunique(), 4, crisis_eval_data["report_date"].nunique()],
    "first_reference_date": [crisis_train_data["report_date"].min(), CRISIS_PURGE_START, crisis_eval_data["report_date"].min()],
    "last_reference_date": [crisis_train_data["report_date"].max(), CRISIS_PURGE_END, crisis_eval_data["report_date"].max()],
    "panel_values_loaded": [True, False, True],
    "count_source": ["recalculated from eligible V2 panel", "calendar only", "recalculated from eligible V2 panel"],
})
cell_03_checks = {
    "Training contains 23 eligible reference quarters from 2001Q2 through 2006Q4":
        crisis_train_data["report_date"].nunique() == 23
        and crisis_train_data["report_date"].min() == CRISIS_TRAIN_START
        and crisis_train_data["report_date"].max() == CRISIS_TRAIN_END,
    "Evaluation contains 20 eligible reference quarters from 2008Q1 through 2012Q4":
        crisis_eval_data["report_date"].nunique() == 20
        and crisis_eval_data["report_date"].min() == CRISIS_EVAL_START
        and crisis_eval_data["report_date"].max() == CRISIS_EVAL_END,
    "The full panel passed Notebook 9 exact-lag checks and every crisis positive has an exact predecessor":
        passed_values(nb09_gate["passed"]).all()
        and crisis_horizon_audit["exact_predecessor"].all()
        and len(crisis_horizon_audit) == training_positives + eval_positives,
    "Calendar 2007 panel values were not loaded":
        not crisis_train_data["report_date"].dt.year.eq(2007).any()
        and not crisis_eval_data["report_date"].dt.year.eq(2007).any(),
    "Crisis keys are unique and the samples are disjoint":
        not crisis_train_data.duplicated(JOIN_KEYS).any()
        and not crisis_eval_data.duplicated(JOIN_KEYS).any()
        and len(crisis_train_data[JOIN_KEYS].merge(crisis_eval_data[JOIN_KEYS], on=JOIN_KEYS)) == 0,
    "Every crisis row receives its immutable Notebook 10 panel_row_id":
        passed_values(nb10_gate["passed"]).all()
        and crisis_train_data["panel_row_id"].notna().all()
        and crisis_eval_data["panel_row_id"].notna().all()
        and not pd.concat([crisis_train_data, crisis_eval_data])["panel_row_id"].duplicated().any(),
    "Counts are derived from Version 2 rows rather than a Version 1 constant":
        crisis_sample_summary.loc[crisis_sample_summary["panel_values_loaded"], "count_source"].str.contains("V2").all(),
    "The audited identity mapping covers every verified CERT transition":
        len(cert_allowlist) == 41
        and all(cert_identity_map[int(row.cert_source)] == cert_identity_map[int(row.cert_reference)]
                for row in cert_allowlist.itertuples(index=False)),
}
notebook_17_cell_03_gate = pd.DataFrame({
    "check": cell_03_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_03_checks.values()],
    "passed": cell_03_checks.values(),
})
crisis_sample_summary.to_csv(NB17_OUTPUT_DIR / "cell_03_crisis_sample_summary.csv", index=False)
cert_identity_audit.to_csv(NB17_OUTPUT_DIR / "cell_03_cert_bootstrap_identity_audit.csv", index=False)
crisis_horizon_audit.to_csv(NB17_OUTPUT_DIR / "cell_03_crisis_exact_predecessor_positive_audit.csv", index=False)
notebook_17_cell_03_gate.to_csv(NB17_OUTPUT_DIR / "cell_03_crisis_sample_gate.csv", index=False)
print("Notebook 17 Cell 3 validation gate:")
display(notebook_17_cell_03_gate)
print("\nRecalculated Version 2 crisis samples:")
display(crisis_sample_summary)
print(f"\nPre-crisis class-weight ratio: {CRISIS_IMBALANCE_RATIO:.6f}")
failed = [name for name, passed in cell_03_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 3 blocked: " + "; ".join(failed))
print("\nCell 3 completed successfully. No Version 1 crisis count was reused.")

In [ ]:
# Fit six fixed crisis models using pre-crisis preprocessing and weights only

if not notebook_17_cell_03_gate["passed"].all():
    raise AssertionError("Cell 4 blocked because Cell 3 did not pass.")


def fit_preprocessor(training, evaluation, predictors, standardise):
    imputer = SimpleImputer(strategy="median")
    x_train = pd.DataFrame(
        imputer.fit_transform(training[predictors]), columns=predictors, index=training.index
    )
    x_eval = pd.DataFrame(
        imputer.transform(evaluation[predictors]), columns=predictors, index=evaluation.index
    )
    if x_train.shape[1] != len(predictors):
        raise AssertionError("A predictor is entirely missing in pre-crisis training.")
    continuous = [predictor for predictor in predictors if predictor != HISTORY_INDICATOR]
    lower = x_train[continuous].quantile(WINSOR_LOWER_QUANTILE)
    upper = x_train[continuous].quantile(WINSOR_UPPER_QUANTILE)
    x_train[continuous] = x_train[continuous].clip(lower=lower, upper=upper, axis=1)
    x_eval[continuous] = x_eval[continuous].clip(lower=lower, upper=upper, axis=1)
    parameters = {
        "predictors": predictors,
        "fit_reference_period": "2001Q2-2006Q4",
        "fit_observations": len(training),
        "fit_positive_bank_quarters": int(training[TARGET].sum()),
        "imputation_medians": dict(zip(predictors, imputer.statistics_.astype(float))),
        "winsor_lower_bounds": lower.to_dict(),
        "winsor_upper_bounds": upper.to_dict(),
        "winsor_excluded": [HISTORY_INDICATOR],
        "standardisation_applied": bool(standardise),
        "purge_values_used": False,
        "evaluation_values_used_for_fit": False,
    }
    if standardise:
        scaler = StandardScaler()
        train_array = scaler.fit_transform(x_train[predictors])
        eval_array = scaler.transform(x_eval[predictors])
        parameters["scaler_means"] = dict(zip(predictors, scaler.mean_.astype(float)))
        parameters["scaler_scales"] = dict(zip(predictors, scaler.scale_.astype(float)))
    else:
        train_array = x_train[predictors].to_numpy(dtype="float32")
        eval_array = x_eval[predictors].to_numpy(dtype="float32")
    if not np.isfinite(train_array).all() or not np.isfinite(eval_array).all():
        raise AssertionError("Training-only preprocessing created a non-finite value.")
    return train_array, eval_array, parameters


def build_crisis_model(family, positive_weight):
    hp = CRISIS_SETTINGS[family]
    if family == "logistic_regression":
        return LogisticRegression(
            penalty=hp["penalty"], C=hp["C"], solver=hp["solver"],
            max_iter=hp["max_iter"], class_weight={0: 1.0, 1: positive_weight},
            random_state=RANDOM_SEED,
        )
    if family == "random_forest":
        return RandomForestClassifier(
            n_estimators=hp["n_estimators"], max_depth=hp["max_depth"],
            min_samples_leaf=hp["min_samples_leaf"], max_features=hp["max_features"],
            bootstrap=True, max_samples=hp["max_samples"],
            class_weight={0: 1.0, 1: positive_weight},
            random_state=RANDOM_SEED, n_jobs=-1,
        )
    if family == "xgboost":
        return XGBClassifier(
            n_estimators=hp["n_estimators"], max_depth=hp["max_depth"],
            learning_rate=hp["learning_rate"], min_child_weight=hp["min_child_weight"],
            subsample=hp["subsample"], colsample_bytree=hp["colsample_bytree"],
            reg_lambda=hp["reg_lambda"], reg_alpha=hp["reg_alpha"],
            scale_pos_weight=positive_weight, objective="binary:logistic",
            eval_metric="logloss", tree_method="hist", random_state=RANDOM_SEED,
            n_jobs=-1, verbosity=0,
        )
    raise ValueError(f"Unknown model family: {family}")


def extract_feature_effects(model, model_id, family, specification, predictors):
    if family == "logistic_regression":
        values = model.coef_[0].astype(float)
        effect_type = "standardised_log_odds_coefficient"
    elif family == "random_forest":
        values = model.feature_importances_.astype(float)
        effect_type = "gini_feature_importance"
    else:
        gain = model.get_booster().get_score(importance_type="gain")
        values = np.array([gain.get(f"f{i}", 0.0) for i in range(len(predictors))], dtype=float)
        if values.sum() > 0:
            values = values / values.sum()
        effect_type = "normalised_gain_feature_importance"
    return pd.DataFrame({
        "model_id": model_id, "model_family": family,
        "specification": specification, "feature": predictors,
        "effect_type": effect_type, "value": values,
    })


y_train = crisis_train_data[TARGET].to_numpy(dtype="int8")
y_eval = crisis_eval_data[TARGET].to_numpy(dtype="int8")
crisis_predictions = {}
fit_rows, metric_rows, feature_parts = [], [], []
for family in MODEL_FAMILIES:
    for specification in SPECIFICATIONS:
        model_id = f"{family}__{specification}"
        predictors = predictor_sets[specification]
        artifact_stem = f"crisis__{model_id}"
        prediction_path = CRISIS_PREDICTION_DIR / f"{artifact_stem}__predictions.parquet"
        model_path = CRISIS_MODEL_DIR / f"{artifact_stem}.joblib"
        preprocessing_path = CRISIS_PREPROCESSING_DIR / f"{artifact_stem}.json"
        effect_path = CRISIS_MODEL_DIR / f"{artifact_stem}__feature_effects.csv"
        checkpoint_path = CRISIS_CHECKPOINT_DIR / f"{artifact_stem}.json"
        signature_payload = {
            "model_id": model_id, "settings": CRISIS_SETTINGS[family],
            "predictors": predictors, "training_rows": len(crisis_train_data),
            "training_positives": training_positives,
            "training_first": str(crisis_train_data["report_date"].min()),
            "training_last": str(crisis_train_data["report_date"].max()),
        }
        signature = hashlib.sha256(
            json.dumps(signature_payload, sort_keys=True).encode("utf-8")
        ).hexdigest()
        artifacts = [prediction_path, model_path, preprocessing_path, effect_path, checkpoint_path]
        present = [path.is_file() for path in artifacts]
        resumed, fit_seconds = False, 0.0
        if any(present) and not all(present):
            raise RuntimeError(f"Partial crisis checkpoint exists for {model_id}.")
        if all(present):
            with open(checkpoint_path, "r", encoding="utf-8") as handle:
                checkpoint = json.load(handle)
            if checkpoint.get("signature") != signature:
                raise AssertionError(f"Stale crisis checkpoint for {model_id}.")
            predictions = pd.read_parquet(prediction_path)
            effects = pd.read_csv(effect_path)
            fit_seconds = float(checkpoint.get("fit_seconds", 0.0))
            resumed = True
        else:
            x_train, x_eval, preprocessing = fit_preprocessor(
                crisis_train_data, crisis_eval_data, predictors,
                standardise=family == "logistic_regression",
            )
            positive_weight = float(
                CRISIS_IMBALANCE_RATIO * CRISIS_SETTINGS[family]["positive_weight_multiplier"]
            )
            model = build_crisis_model(family, positive_weight)
            started = time.perf_counter()
            model.fit(x_train, y_train)
            fit_seconds = time.perf_counter() - started
            probability = model.predict_proba(x_eval)[:, 1].astype("float64")
            predictions = crisis_eval_data[
                ["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET]
            ].copy()
            predictions["predicted_probability"] = probability.astype("float32")
            predictions["model_id"] = model_id
            predictions["model_family"] = family
            predictions["specification"] = specification
            predictions["analysis_status"] = "exploratory crisis robustness"
            effects = extract_feature_effects(
                model, model_id, family, specification, predictors
            )
            preprocessing.update({
                "model_id": model_id,
                "fixed_hyperparameters": CRISIS_SETTINGS[family],
                "pre_crisis_negative_to_positive_ratio": CRISIS_IMBALANCE_RATIO,
                "positive_class_weight": positive_weight,
                "settings_source": "fixed in Notebook 17; independent of Notebook 15 tuning",
            })
            predictions.to_parquet(prediction_path, index=False, compression="snappy")
            joblib.dump(model, model_path, compress=3)
            strict_json_dump(preprocessing, preprocessing_path)
            effects.to_csv(effect_path, index=False)
            strict_json_dump({
                "signature": signature, "model_id": model_id,
                "fit_seconds": fit_seconds, "status": "COMPLETE",
            }, checkpoint_path)

        predictions["report_date"] = pd.to_datetime(
            predictions["report_date"], errors="raise"
        ).dt.normalize()
        for column in ["cert", "idrssd", "bootstrap_identity"]:
            predictions[column] = pd.to_numeric(
                predictions[column], errors="raise"
            ).astype("int64")
        predictions[TARGET] = pd.to_numeric(
            predictions[TARGET], errors="raise"
        ).astype("int8")
        expected = crisis_eval_data[
            ["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET]
        ].copy()
        identity_columns = ["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET]
        if predictions["panel_row_id"].duplicated().any():
            raise AssertionError(f"Duplicate crisis prediction panel_row_id for {model_id}.")
        predictions.sort_values("panel_row_id", kind="mergesort", inplace=True)
        predictions.reset_index(drop=True, inplace=True)
        expected.sort_values("panel_row_id", kind="mergesort", inplace=True)
        expected.reset_index(drop=True, inplace=True)
        try:
            pd.testing.assert_frame_equal(
                predictions[identity_columns], expected[identity_columns],
                check_dtype=False, check_exact=True,
            )
        except AssertionError as exc:
            missing_ids = expected.loc[
                ~expected["panel_row_id"].isin(predictions["panel_row_id"]), "panel_row_id"
            ].head(10).tolist()
            extra_ids = predictions.loc[
                ~predictions["panel_row_id"].isin(expected["panel_row_id"]), "panel_row_id"
            ].head(10).tolist()
            raise AssertionError(
                f"Crisis prediction identity mismatch for {model_id}; "
                f"missing panel_row_id examples={missing_ids}, extra examples={extra_ids}."
            ) from exc
        score = predictions["predicted_probability"].to_numpy(dtype="float64")
        if not np.isfinite(score).all() or not pd.Series(score).between(0, 1).all():
            raise AssertionError(f"Invalid crisis score for {model_id}.")
        crisis_predictions[model_id] = predictions
        pr_auc = float(average_precision_score(y_eval, score))
        metric_rows.append({
            "model_id": model_id, "model_family": family,
            "specification": specification,
            "evaluation_observations": len(y_eval),
            "positive_bank_quarters": int(y_eval.sum()),
            "distinct_failed_banks": eval_failed_banks,
            "prevalence": float(y_eval.mean()),
            "roc_auc": float(roc_auc_score(y_eval, score)),
            "pr_auc": pr_auc,
            "pr_auc_divided_by_prevalence": pr_auc / y_eval.mean(),
            "brier_score_secondary": float(brier_score_loss(y_eval, score)),
            "log_loss_secondary": float(log_loss(y_eval, np.clip(score, 1e-12, 1 - 1e-12))),
            "used_for_model_selection": False,
        })
        fit_rows.append({
            "model_id": model_id, "model_family": family,
            "specification": specification, "predictor_count": len(predictors),
            "training_observations": len(y_train),
            "training_positive_bank_quarters": int(y_train.sum()),
            "training_distinct_failed_banks": training_failed_banks,
            "positive_class_weight": CRISIS_IMBALANCE_RATIO,
            "fit_seconds": fit_seconds, "resumed": resumed,
            "settings_source": "Notebook 17 fixed pre-specification; not Notebook 15",
        })
        feature_parts.append(effects)
        print(f"Complete: {model_id} ({'resumed' if resumed else 'fit'})")

# A training-only low-equity baseline is carried through the identical crisis denominator.
baseline_feature = "equity_to_assets_lag1"
baseline_median = float(crisis_train_data[baseline_feature].median())
baseline_imputed_train = crisis_train_data[baseline_feature].fillna(baseline_median)
baseline_lower = float(baseline_imputed_train.quantile(WINSOR_LOWER_QUANTILE))
baseline_upper = float(baseline_imputed_train.quantile(WINSOR_UPPER_QUANTILE))
baseline_value = crisis_eval_data[baseline_feature].fillna(baseline_median).clip(
    lower=baseline_lower, upper=baseline_upper
)
crisis_baseline = crisis_eval_data[
    ["panel_row_id", *PANEL_KEYS, "bootstrap_identity", TARGET]
].copy()
crisis_baseline["risk_score"] = -baseline_value.astype("float64")

crisis_fit_register = pd.DataFrame(fit_rows)
crisis_point_metrics = pd.DataFrame(metric_rows)
crisis_feature_effects = pd.concat(feature_parts, ignore_index=True)
crisis_fit_register.to_csv(NB17_OUTPUT_DIR / "cell_04_crisis_fit_register.csv", index=False)
crisis_point_metrics.to_csv(NB17_OUTPUT_DIR / "crisis_point_metrics.csv", index=False)
crisis_feature_effects.to_csv(NB17_OUTPUT_DIR / "crisis_model_feature_effects.csv", index=False)
strict_json_dump({
    "feature": baseline_feature, "fit_period": "2001Q2-2006Q4",
    "training_median": baseline_median, "winsor_lower": baseline_lower,
    "winsor_upper": baseline_upper, "higher_risk_definition": "negative winsorised equity-to-assets",
}, CRISIS_PREPROCESSING_DIR / "crisis__low_equity_baseline.json")
cell_04_checks = {
    "Exactly six fixed crisis specifications were fitted or resumed":
        len(crisis_fit_register) == 6 and set(crisis_fit_register["model_id"]) == set(MODEL_IDS),
    "Every fit uses only recalculated pre-crisis Version 2 rows and labels":
        crisis_fit_register["training_observations"].eq(len(crisis_train_data)).all()
        and crisis_fit_register["training_positive_bank_quarters"].eq(training_positives).all(),
    "Every model uses the pre-crisis class-weight ratio with neutral multiplier":
        np.allclose(crisis_fit_register["positive_class_weight"], CRISIS_IMBALANCE_RATIO),
    "All six predictions are finite on the same eligible crisis rows":
        len(crisis_predictions) == 6
        and all(len(frame) == len(crisis_eval_data) for frame in crisis_predictions.values()),
    "Crisis metrics are finite and never used for model selection":
        np.isfinite(crisis_point_metrics[["roc_auc", "pr_auc", "brier_score_secondary", "log_loss_secondary"]]).all().all()
        and not crisis_point_metrics["used_for_model_selection"].any(),
    "The low-equity baseline uses training-only median and winsor bounds":
        np.isfinite([baseline_median, baseline_lower, baseline_upper]).all()
        and len(crisis_baseline) == len(crisis_eval_data),
}
notebook_17_cell_04_gate = pd.DataFrame({
    "check": cell_04_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_04_checks.values()],
    "passed": cell_04_checks.values(),
})
notebook_17_cell_04_gate.to_csv(NB17_OUTPUT_DIR / "cell_04_crisis_fit_gate.csv", index=False)
print("Notebook 17 Cell 4 validation gate:")
display(notebook_17_cell_04_gate)
print("\nExploratory crisis ranking metrics and secondary probability diagnostics:")
display(crisis_point_metrics)
failed = [name for name, passed in cell_04_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 4 blocked: " + "; ".join(failed))
print("\nCell 4 completed successfully. Crisis outcomes changed no locked final-test decision.")

In [ ]:
# Calculate crisis quarterly capacity with explicit total and per-quarter denominators

if not notebook_17_cell_04_gate["passed"].all():
    raise AssertionError("Cell 5 blocked because Cell 4 did not pass.")


def build_quarterly_capacity(frame, source_id, scope, denominator_label):
    base = frame[[
        "report_date", "cert", "idrssd", "bootstrap_identity", TARGET, "risk_score"
    ]].copy().reset_index(drop=True)
    base["_analysis_row_id"] = np.arange(len(base), dtype="int64")
    if base.duplicated(JOIN_KEYS).any():
        raise AssertionError(f"Duplicate comparison keys for {scope}/{source_id}.")
    ranked = base.sort_values(
        ["report_date", "risk_score", "cert", "idrssd"],
        ascending=[True, False, True, True], kind="mergesort",
    ).copy()
    ranked["quarter_rank"] = ranked.groupby("report_date").cumcount() + 1
    ranked["active_banks"] = ranked.groupby("report_date")["idrssd"].transform("size")
    total_positive = int(base[TARGET].sum())
    total_failed = int(base.loc[base[TARGET].eq(1), "bootstrap_identity"].nunique())
    evaluation_quarters = int(base["report_date"].nunique())
    prevalence = float(base[TARGET].mean())
    summary_rows, quarterly_rows, membership = [], [], {}
    for percentage in TOP_RISK_PERCENTAGES:
        group_size = np.maximum(1, np.ceil(ranked["active_banks"] * percentage)).astype(int)
        ranked["flagged"] = ranked["quarter_rank"].le(group_size)
        aligned_flag = ranked.sort_values("_analysis_row_id", kind="mergesort")["flagged"].to_numpy(bool)
        membership[percentage] = aligned_flag
        for report_date, quarter in ranked.groupby("report_date", sort=True):
            flagged = quarter.loc[quarter["flagged"]]
            quarterly_rows.append({
                "scope": scope, "source_id": source_id,
                "denominator_label": denominator_label,
                "report_date": report_date, "top_risk_percentage": percentage,
                "active_banks_quarterly_denominator": len(quarter),
                "flagged_bank_quarters_in_quarter": len(flagged),
                "positive_bank_quarters_in_quarter": int(quarter[TARGET].sum()),
                "positive_bank_quarters_flagged_in_quarter": int(flagged[TARGET].sum()),
                "distinct_failed_banks_in_quarter": int(quarter.loc[
                    quarter[TARGET].eq(1), "bootstrap_identity"
                ].nunique()),
                "distinct_failed_banks_captured_in_quarter": int(flagged.loc[
                    flagged[TARGET].eq(1), "bootstrap_identity"
                ].nunique()),
            })
        flagged = base.loc[aligned_flag]
        positive_captured = int(flagged[TARGET].sum())
        failed_captured = int(flagged.loc[
            flagged[TARGET].eq(1), "bootstrap_identity"
        ].nunique())
        total_flagged = int(aligned_flag.sum())
        precision = positive_captured / total_flagged
        summary_rows.append({
            "scope": scope, "source_id": source_id,
            "denominator_label": denominator_label,
            "top_risk_percentage": percentage,
            "evaluation_observations": len(base),
            "evaluation_quarters": evaluation_quarters,
            "evaluation_prevalence": prevalence,
            "flagged_bank_quarters_total_across_evaluation": total_flagged,
            "mean_flagged_banks_per_evaluation_quarter": total_flagged / evaluation_quarters,
            "percentage_of_bank_quarters_flagged": total_flagged / len(base),
            "positive_bank_quarters_total": total_positive,
            "positive_bank_quarters_captured": positive_captured,
            "positive_bank_quarter_capture_rate": positive_captured / total_positive,
            "distinct_failed_banks_total": total_failed,
            "distinct_failed_banks_captured": failed_captured,
            "distinct_failed_bank_capture_rate": failed_captured / total_failed,
            "precision_in_top_group": precision,
            "lift_over_prevalence": precision / prevalence,
            "primary_capacity_result": "distinct_failed_bank_capture_rate",
            "secondary_capacity_result": "positive_bank_quarter_capture_rate",
        })
    return pd.DataFrame(summary_rows), pd.DataFrame(quarterly_rows), membership


crisis_risk_frames = {}
for model_id, predictions in crisis_predictions.items():
    frame = predictions[["report_date", "cert", "idrssd", "bootstrap_identity", TARGET]].copy()
    frame["risk_score"] = predictions["predicted_probability"].astype("float64")
    crisis_risk_frames[model_id] = frame
crisis_risk_frames[BASELINE_ID] = crisis_baseline

capacity_parts, quarterly_parts, crisis_membership = [], [], {}
for source_id, frame in crisis_risk_frames.items():
    summary, quarterly, membership = build_quarterly_capacity(
        frame, source_id, "crisis_evaluation",
        "eligible V2 active banks within each 2008Q1-2012Q4 quarter",
    )
    capacity_parts.append(summary)
    quarterly_parts.append(quarterly)
    for percentage, flags in membership.items():
        crisis_membership[(source_id, percentage)] = flags
crisis_capacity_point = pd.concat(capacity_parts, ignore_index=True)
crisis_quarterly_capacity = pd.concat(quarterly_parts, ignore_index=True)
crisis_capacity_point.to_csv(NB17_OUTPUT_DIR / "crisis_capacity_point_results.csv", index=False)
crisis_quarterly_capacity.to_csv(NB17_OUTPUT_DIR / "crisis_quarterly_capacity_results.csv", index=False)
cell_05_checks = {
    "Six crisis models and the baseline produce 28 capacity rows":
        len(crisis_capacity_point) == 28 and set(crisis_capacity_point["source_id"]) == set(crisis_risk_frames),
    "Every source contains exactly four predeclared capacity fractions":
        crisis_capacity_point.groupby("source_id")["top_risk_percentage"].apply(
            lambda values: set(values) == set(TOP_RISK_PERCENTAGES)
        ).all(),
    "Quarterly results contain 7 sources by 4 fractions by 20 evaluation quarters":
        len(crisis_quarterly_capacity) == 7 * 4 * 20,
    "Flagged counts are reported both as evaluation totals and per-quarter means":
        np.allclose(
            crisis_capacity_point["flagged_bank_quarters_total_across_evaluation"],
            crisis_capacity_point["mean_flagged_banks_per_evaluation_quarter"]
            * crisis_capacity_point["evaluation_quarters"],
        ),
    "Primary and secondary capture rates are complete and bounded":
        crisis_capacity_point[[
            "distinct_failed_bank_capture_rate", "positive_bank_quarter_capture_rate"
        ]].apply(lambda values: values.between(0, 1).all()).all(),
}
notebook_17_cell_05_gate = pd.DataFrame({
    "check": cell_05_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_05_checks.values()],
    "passed": cell_05_checks.values(),
})
notebook_17_cell_05_gate.to_csv(NB17_OUTPUT_DIR / "cell_05_crisis_capacity_gate.csv", index=False)
print("Notebook 17 Cell 5 validation gate:")
display(notebook_17_cell_05_gate)
print("\nExploratory crisis capacity (totals and mean flagged banks per quarter):")
display(crisis_capacity_point)
failed = [name for name, passed in cell_05_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 5 blocked: " + "; ".join(failed))
print("\nCell 5 completed successfully.")

In [ ]:
# Add audited-identity clustered intervals to the crisis analysis

if not notebook_17_cell_05_gate["passed"].all():
    raise AssertionError("Cell 6 blocked because Cell 5 did not pass.")


def prepare_weighted_discrimination(y, score):
    order = np.argsort(-score, kind="mergesort")
    sorted_score = score[order]
    sorted_y = y[order]
    starts = np.r_[0, np.flatnonzero(sorted_score[1:] != sorted_score[:-1]) + 1]
    return order, sorted_y, starts


def weighted_auc_and_ap(prepared, row_weights):
    order, sorted_y, starts = prepared
    sorted_weight = row_weights[order].astype("float64")
    positive_group = np.add.reduceat(sorted_weight * sorted_y, starts)
    negative_group = np.add.reduceat(sorted_weight * (1 - sorted_y), starts)
    total_positive, total_negative = positive_group.sum(), negative_group.sum()
    if total_positive <= 0 or total_negative <= 0:
        return np.nan, np.nan
    cumulative_positive = np.cumsum(positive_group)
    cumulative_negative = np.cumsum(negative_group)
    true_positive_rate = np.r_[0, cumulative_positive / total_positive]
    false_positive_rate = np.r_[0, cumulative_negative / total_negative]
    roc_auc = float(
        np.trapezoid(true_positive_rate, false_positive_rate)
        if hasattr(np, "trapezoid") else np.trapz(true_positive_rate, false_positive_rate)
    )
    denominator = cumulative_positive + cumulative_negative
    precision = np.divide(
        cumulative_positive, denominator,
        out=np.zeros_like(cumulative_positive), where=denominator > 0,
    )
    pr_auc = float(np.sum((positive_group / total_positive) * precision))
    return roc_auc, pr_auc


def common_cluster_bootstrap(source_frames, membership, point_capacity, seed, scope):
    source_ids = list(source_frames)
    reference = source_frames[source_ids[0]].reset_index(drop=True)
    key_columns = ["report_date", "idrssd", "bootstrap_identity", TARGET]
    for source_id, frame in source_frames.items():
        candidate = frame.reset_index(drop=True)
        if not candidate[key_columns].equals(reference[key_columns]):
            raise AssertionError(f"Bootstrap rows disagree for {scope}/{source_id}.")
    y = reference[TARGET].to_numpy(dtype="int8")
    bank_ids = np.sort(reference["bootstrap_identity"].unique())
    bank_code = pd.Categorical(
        reference["bootstrap_identity"], categories=bank_ids
    ).codes.astype("int32")
    n_banks = len(bank_ids)
    rng = np.random.default_rng(seed)
    bootstrap_counts = np.empty((BOOTSTRAP_REPLICATES, n_banks), dtype="int16")
    probabilities = np.full(n_banks, 1 / n_banks)
    for replicate in range(BOOTSTRAP_REPLICATES):
        bootstrap_counts[replicate] = rng.multinomial(n_banks, probabilities).astype("int16")
    alpha = (1 - BOOTSTRAP_CONFIDENCE_LEVEL) / 2
    discrimination_replicates, discrimination_rows = [], []
    for source_id, frame in source_frames.items():
        score = frame["risk_score"].to_numpy(dtype="float64")
        prepared = prepare_weighted_discrimination(y, score)
        point_roc = float(roc_auc_score(y, score))
        point_pr = float(average_precision_score(y, score))
        check_roc, check_pr = weighted_auc_and_ap(prepared, np.ones(len(y)))
        if not np.allclose([point_roc, point_pr], [check_roc, check_pr], atol=1e-12):
            raise AssertionError(f"Weighted metric implementation disagrees for {scope}/{source_id}.")
        current = []
        for replicate in range(BOOTSTRAP_REPLICATES):
            roc_value, pr_value = weighted_auc_and_ap(
                prepared, bootstrap_counts[replicate, bank_code]
            )
            row = {
                "scope": scope, "source_id": source_id, "replicate": replicate + 1,
                "roc_auc": roc_value, "pr_auc": pr_value,
            }
            current.append(row)
            discrimination_replicates.append(row)
        current = pd.DataFrame(current)
        discrimination_rows.append({
            "scope": scope, "source_id": source_id,
            "observations": len(y), "positive_bank_quarters": int(y.sum()),
            "distinct_failed_banks": int(reference.loc[
                reference[TARGET].eq(1), "bootstrap_identity"
            ].nunique()),
            "prevalence": float(y.mean()),
            "roc_auc": point_roc,
            "roc_auc_ci_lower": current["roc_auc"].quantile(alpha),
            "roc_auc_ci_upper": current["roc_auc"].quantile(1 - alpha),
            "pr_auc": point_pr,
            "pr_auc_ci_lower": current["pr_auc"].quantile(alpha),
            "pr_auc_ci_upper": current["pr_auc"].quantile(1 - alpha),
            "pr_auc_divided_by_prevalence": point_pr / y.mean(),
            "bootstrap_identity": "audited CERT component",
            "bootstrap_seed": seed,
            "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        })

    total_by_bank = np.bincount(bank_code, minlength=n_banks).astype("float64")
    positive_by_bank = np.bincount(bank_code, weights=y, minlength=n_banks).astype("float64")
    failed_bank = positive_by_bank > 0
    capacity_replicates, capacity_rows = [], []
    for source_id in source_ids:
        for percentage in TOP_RISK_PERCENTAGES:
            flag = membership[(source_id, percentage)]
            flagged_by_bank = np.bincount(
                bank_code, weights=flag.astype(float), minlength=n_banks
            )
            flagged_positive_by_bank = np.bincount(
                bank_code, weights=(flag & y.astype(bool)).astype(float), minlength=n_banks
            )
            captured_failed_bank = flagged_positive_by_bank > 0
            current = []
            for replicate in range(BOOTSTRAP_REPLICATES):
                multiplicity = bootstrap_counts[replicate].astype("float64")
                observations = multiplicity @ total_by_bank
                positives = multiplicity @ positive_by_bank
                failed = multiplicity @ failed_bank.astype(float)
                flagged = multiplicity @ flagged_by_bank
                flagged_positive = multiplicity @ flagged_positive_by_bank
                captured = multiplicity @ captured_failed_bank.astype(float)
                precision = flagged_positive / flagged if flagged else np.nan
                prevalence = positives / observations if observations else np.nan
                row = {
                    "scope": scope, "source_id": source_id,
                    "top_risk_percentage": percentage, "replicate": replicate + 1,
                    "percentage_of_bank_quarters_flagged": flagged / observations,
                    "precision_in_top_group": precision,
                    "lift_over_prevalence": precision / prevalence if prevalence else np.nan,
                    "distinct_failed_bank_capture_rate": captured / failed if failed else np.nan,
                    "positive_bank_quarter_capture_rate": flagged_positive / positives if positives else np.nan,
                }
                current.append(row)
                capacity_replicates.append(row)
            current = pd.DataFrame(current)
            point = point_capacity.loc[
                point_capacity["source_id"].eq(source_id)
                & point_capacity["top_risk_percentage"].eq(percentage)
            ].iloc[0].to_dict()
            for metric in [
                "percentage_of_bank_quarters_flagged", "precision_in_top_group",
                "lift_over_prevalence", "distinct_failed_bank_capture_rate",
                "positive_bank_quarter_capture_rate",
            ]:
                point[f"{metric}_ci_lower"] = current[metric].quantile(alpha)
                point[f"{metric}_ci_upper"] = current[metric].quantile(1 - alpha)
            point["bootstrap_identity"] = "audited CERT component"
            point["bootstrap_seed"] = seed
            point["bootstrap_replicates"] = BOOTSTRAP_REPLICATES
            capacity_rows.append(point)
    return (
        pd.DataFrame(discrimination_replicates), pd.DataFrame(discrimination_rows),
        pd.DataFrame(capacity_replicates), pd.DataFrame(capacity_rows), bootstrap_counts,
    )


(
    crisis_discrimination_replicates, crisis_discrimination_with_ci,
    crisis_capacity_replicates, crisis_capacity_with_ci, crisis_bootstrap_counts,
) = common_cluster_bootstrap(
    crisis_risk_frames, crisis_membership, crisis_capacity_point,
    BOOTSTRAP_SEED, "crisis_evaluation",
)
crisis_discrimination_replicates.to_parquet(
    NB17_OUTPUT_DIR / "crisis_discrimination_cluster_bootstrap_replicates.parquet",
    index=False, compression="snappy",
)
crisis_capacity_replicates.to_parquet(
    NB17_OUTPUT_DIR / "crisis_capacity_cluster_bootstrap_replicates.parquet",
    index=False, compression="snappy",
)
crisis_discrimination_with_ci.to_csv(
    NB17_OUTPUT_DIR / "crisis_discrimination_with_clustered_ci.csv", index=False
)
crisis_capacity_with_ci.to_csv(
    NB17_OUTPUT_DIR / "crisis_capacity_with_clustered_ci.csv", index=False
)
cell_06_checks = {
    "The crisis bootstrap contains exactly 1,000 audited-identity resamples":
        crisis_bootstrap_counts.shape[0] == BOOTSTRAP_REPLICATES
        and (crisis_bootstrap_counts.sum(axis=1) == crisis_bootstrap_counts.shape[1]).all(),
    "All seven sources have 1,000 complete discrimination replicates":
        len(crisis_discrimination_replicates) == 7 * BOOTSTRAP_REPLICATES
        and crisis_discrimination_replicates.groupby("source_id")["replicate"].nunique().eq(BOOTSTRAP_REPLICATES).all(),
    "All 28 crisis capacity rows have clustered intervals":
        len(crisis_capacity_replicates) == 28 * BOOTSTRAP_REPLICATES
        and len(crisis_capacity_with_ci) == 28,
    "Every crisis interval uses the Version 1-identical seed and identity rule":
        crisis_discrimination_with_ci["bootstrap_seed"].eq(BOOTSTRAP_SEED).all()
        and crisis_capacity_with_ci["bootstrap_identity"].eq("audited CERT component").all(),
    "All bounded point estimates are valid":
        crisis_discrimination_with_ci[["roc_auc", "pr_auc"]].apply(
            lambda values: values.between(0, 1).all()
        ).all()
        and crisis_capacity_with_ci[[
            "distinct_failed_bank_capture_rate", "positive_bank_quarter_capture_rate"
        ]].apply(lambda values: values.between(0, 1).all()).all(),
}
notebook_17_cell_06_gate = pd.DataFrame({
    "check": cell_06_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_06_checks.values()],
    "passed": cell_06_checks.values(),
})
notebook_17_cell_06_gate.to_csv(NB17_OUTPUT_DIR / "cell_06_crisis_bootstrap_gate.csv", index=False)
print("Notebook 17 Cell 6 validation gate:")
display(notebook_17_cell_06_gate)
print("\nExploratory crisis discrimination with clustered intervals:")
display(crisis_discrimination_with_ci)
failed = [name for name, passed in cell_06_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 6 blocked: " + "; ".join(failed))
print("\nCell 6 completed successfully.")

In [ ]:
# Compare archived Version 1 and Version 2 without refitting either final-test model

if not notebook_17_cell_06_gate["passed"].all():
    raise AssertionError("Cell 7 blocked because Cell 6 did not pass.")


def load_final_prediction(path, model_id, version):
    frame = pd.read_parquet(path)
    required = {"report_date", "cert", "idrssd", TARGET, "predicted_probability"}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise KeyError(f"{version} {model_id} prediction file missing: {', '.join(missing)}")
    frame = frame[["report_date", "cert", "idrssd", TARGET, "predicted_probability"]].copy()
    frame["report_date"] = pd.to_datetime(frame["report_date"], errors="raise").dt.normalize()
    frame["cert"] = pd.to_numeric(frame["cert"], errors="raise").astype("int64")
    frame["idrssd"] = pd.to_numeric(frame["idrssd"], errors="raise").astype("int64")
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors="raise").astype("int8")
    frame["predicted_probability"] = pd.to_numeric(
        frame["predicted_probability"], errors="raise"
    ).astype("float64")
    frame["bootstrap_identity"] = frame["cert"].map(cert_identity_map).fillna(frame["cert"]).astype("int64")
    frame.sort_values(JOIN_KEYS, kind="mergesort", inplace=True)
    frame.reset_index(drop=True, inplace=True)
    if frame.duplicated(JOIN_KEYS).any():
        raise AssertionError(f"Duplicate {version} final-test key for {model_id}.")
    if not frame["predicted_probability"].between(0, 1).all():
        raise AssertionError(f"Invalid {version} final-test probability for {model_id}.")
    return frame


v2_final_predictions = {
    model_id: load_final_prediction(path, model_id, "Version 2")
    for model_id, path in V2_PREDICTION_PATHS.items()
}
v1_full_predictions = {
    model_id: load_final_prediction(path, model_id, "archived Version 1")
    for model_id, path in V1_PREDICTION_PATHS.items()
}

v2_reference = v2_final_predictions[MODEL_IDS[0]].copy()
for model_id, frame in v2_final_predictions.items():
    if not frame[[*JOIN_KEYS, "cert", TARGET, "bootstrap_identity"]].equals(
        v2_reference[[*JOIN_KEYS, "cert", TARGET, "bootstrap_identity"]]
    ):
        raise AssertionError(f"Version 2 rows disagree across models for {model_id}.")
v1_reference = v1_full_predictions[MODEL_IDS[0]].copy()
for model_id, frame in v1_full_predictions.items():
    if not frame[[*JOIN_KEYS, "cert", TARGET, "bootstrap_identity"]].equals(
        v1_reference[[*JOIN_KEYS, "cert", TARGET, "bootstrap_identity"]]
    ):
        raise AssertionError(f"Archived Version 1 rows disagree across models for {model_id}.")

v2_key_frame = v2_reference[[
    "report_date", "idrssd", "cert", "bootstrap_identity", TARGET
]].rename(columns={"cert": "v2_cert", TARGET: "v2_target"})
v1_restricted_predictions = {}
loss_rows, cert_comparison_rows = [], []
for model_id, v1_frame in v1_full_predictions.items():
    merged = v2_key_frame.merge(
        v1_frame[[
            "report_date", "idrssd", "cert", "bootstrap_identity", TARGET, "predicted_probability"
        ]].rename(columns={
            "cert": "v1_cert", "bootstrap_identity": "v1_bootstrap_identity",
            TARGET: "v1_target",
        }),
        on=JOIN_KEYS, how="left", validate="one_to_one", indicator=True,
    )
    if not merged["_merge"].eq("both").all():
        missing = merged.loc[merged["_merge"].ne("both"), JOIN_KEYS].head()
        raise AssertionError(
            f"Archived Version 1 lacks Version 2-eligible keys for {model_id}:\n{missing}"
        )
    if not merged["v1_target"].astype("int8").eq(merged["v2_target"].astype("int8")).all():
        raise AssertionError(f"Version 1 and Version 2 targets disagree on overlapping keys for {model_id}.")
    restricted = merged[[
        "report_date", "v2_cert", "idrssd", "bootstrap_identity", "v2_target",
        "predicted_probability",
    ]].rename(columns={"v2_cert": "cert", "v2_target": TARGET})
    restricted["cert"] = restricted["cert"].astype("int64")
    restricted["bootstrap_identity"] = restricted["bootstrap_identity"].astype("int64")
    restricted[TARGET] = restricted[TARGET].astype("int8")
    v1_restricted_predictions[model_id] = restricted

    full_positive_identities = set(v1_frame.loc[
        v1_frame[TARGET].eq(1), "bootstrap_identity"
    ].astype(int))
    restricted_positive_identities = set(restricted.loc[
        restricted[TARGET].eq(1), "bootstrap_identity"
    ].astype(int))
    loss_rows.append({
        "model_id": model_id,
        "version_1_full_rows": len(v1_frame),
        "version_1_restricted_rows": len(restricted),
        "rows_lost_under_v2_eligibility": len(v1_frame) - len(restricted),
        "version_1_full_positive_bank_quarters": int(v1_frame[TARGET].sum()),
        "version_1_restricted_positive_bank_quarters": int(restricted[TARGET].sum()),
        "positive_bank_quarters_lost_under_v2_eligibility": int(v1_frame[TARGET].sum() - restricted[TARGET].sum()),
        "version_1_full_distinct_failed_banks": len(full_positive_identities),
        "version_1_restricted_distinct_failed_banks": len(restricted_positive_identities),
        "distinct_failed_banks_lost_under_v2_eligibility": len(full_positive_identities - restricted_positive_identities),
    })
    cert_comparison_rows.append({
        "model_id": model_id,
        "overlapping_rows": len(merged),
        "source_cert_differs_between_versions": int(merged["v1_cert"].ne(merged["v2_cert"]).sum()),
        "bootstrap_identity_source": "Version 2 audited CERT component on common eligible keys",
    })

eligibility_losses = pd.DataFrame(loss_rows)
cert_key_audit = pd.DataFrame(cert_comparison_rows)
if eligibility_losses.drop(columns="model_id").nunique().max() != 1:
    raise AssertionError("Eligibility losses differ across archived Version 1 model files.")


def as_risk_frames(prediction_frames, prefix):
    result = {}
    for model_id, predictions in prediction_frames.items():
        frame = predictions[[
            "report_date", "cert", "idrssd", "bootstrap_identity", TARGET
        ]].copy().reset_index(drop=True)
        frame["risk_score"] = predictions["predicted_probability"].astype("float64")
        result[f"{prefix}__{model_id}"] = frame
    return result


v1_full_risk = as_risk_frames(v1_full_predictions, "v1_full")
v1_restricted_risk = as_risk_frames(v1_restricted_predictions, "v1_restricted")
v2_like_risk = as_risk_frames(v2_final_predictions, "v2_main")
like_risk = {**v1_restricted_risk, **v2_like_risk}

comparison_capacity_parts, comparison_quarterly_parts = [], []
v1_full_membership, like_membership = {}, {}
for source_id, frame in v1_full_risk.items():
    summary, quarterly, memberships = build_quarterly_capacity(
        frame, source_id, "v1_full_non_like_for_like",
        "archived Version 1 eligible active banks within each original test quarter",
    )
    comparison_capacity_parts.append(summary)
    comparison_quarterly_parts.append(quarterly)
    for percentage, flags in memberships.items():
        v1_full_membership[(source_id, percentage)] = flags
for source_id, frame in like_risk.items():
    summary, quarterly, memberships = build_quarterly_capacity(
        frame, source_id, "like_for_like_v2_denominator",
        "same Version 2-eligible active banks within each 2020Q1-2025Q2 quarter",
    )
    comparison_capacity_parts.append(summary)
    comparison_quarterly_parts.append(quarterly)
    for percentage, flags in memberships.items():
        like_membership[(source_id, percentage)] = flags
comparison_capacity_point = pd.concat(comparison_capacity_parts, ignore_index=True)
comparison_quarterly_capacity = pd.concat(comparison_quarterly_parts, ignore_index=True)

(
    v1_full_discrimination_replicates, v1_full_discrimination_ci,
    v1_full_capacity_replicates, v1_full_capacity_ci, v1_full_bootstrap_counts,
) = common_cluster_bootstrap(
    v1_full_risk, v1_full_membership,
    comparison_capacity_point.loc[comparison_capacity_point["scope"].eq("v1_full_non_like_for_like")],
    BOOTSTRAP_SEED, "v1_full_non_like_for_like",
)
(
    like_discrimination_replicates, like_discrimination_ci,
    like_capacity_replicates, like_capacity_ci, like_bootstrap_counts,
) = common_cluster_bootstrap(
    like_risk, like_membership,
    comparison_capacity_point.loc[comparison_capacity_point["scope"].eq("like_for_like_v2_denominator")],
    BOOTSTRAP_SEED, "like_for_like_v2_denominator",
)

comparison_discrimination = pd.concat([
    v1_full_discrimination_ci, like_discrimination_ci
], ignore_index=True)
comparison_capacity_with_ci = pd.concat([
    v1_full_capacity_ci, like_capacity_ci
], ignore_index=True)
comparison_discrimination["version"] = np.select([
    comparison_discrimination["source_id"].str.startswith("v1_full__"),
    comparison_discrimination["source_id"].str.startswith("v1_restricted__"),
], ["Version 1 archived", "Version 1 archived"], default="Version 2 main")
comparison_discrimination["sample_definition"] = np.select([
    comparison_discrimination["source_id"].str.startswith("v1_full__"),
    comparison_discrimination["source_id"].str.startswith("v1_restricted__"),
], ["full original Version 1 sample", "restricted to Version 2 eligible keys"],
   default="Version 2 eligible keys")
comparison_discrimination["like_for_like"] = comparison_discrimination["scope"].eq(
    "like_for_like_v2_denominator"
)
comparison_discrimination["analysis_priority"] = np.where(
    comparison_discrimination["version"].eq("Version 2 main"),
    "main analysis", "secondary sensitivity evidence",
)
comparison_discrimination["causal_interpretation_permitted"] = False
comparison_discrimination["model_id"] = comparison_discrimination["source_id"].str.replace(
    r"^(v1_full|v1_restricted|v2_main)__", "", regex=True
)

comparison_capacity_with_ci["version"] = np.select([
    comparison_capacity_with_ci["source_id"].str.startswith("v1_full__"),
    comparison_capacity_with_ci["source_id"].str.startswith("v1_restricted__"),
], ["Version 1 archived", "Version 1 archived"], default="Version 2 main")
comparison_capacity_with_ci["sample_definition"] = np.select([
    comparison_capacity_with_ci["source_id"].str.startswith("v1_full__"),
    comparison_capacity_with_ci["source_id"].str.startswith("v1_restricted__"),
], ["full original Version 1 sample", "restricted to Version 2 eligible keys"],
   default="Version 2 eligible keys")
comparison_capacity_with_ci["like_for_like"] = comparison_capacity_with_ci["scope"].eq(
    "like_for_like_v2_denominator"
)
comparison_capacity_with_ci["model_id"] = comparison_capacity_with_ci["source_id"].str.replace(
    r"^(v1_full|v1_restricted|v2_main)__", "", regex=True
)

like_quarterly = comparison_quarterly_capacity.loc[
    comparison_quarterly_capacity["scope"].eq("like_for_like_v2_denominator")
].copy()
group_size_audit_rows = []
for model_id in MODEL_IDS:
    for percentage in TOP_RISK_PERCENTAGES:
        v1_counts = like_quarterly.loc[
            like_quarterly["source_id"].eq(f"v1_restricted__{model_id}")
            & like_quarterly["top_risk_percentage"].eq(percentage),
            ["report_date", "active_banks_quarterly_denominator", "flagged_bank_quarters_in_quarter"],
        ].sort_values("report_date").reset_index(drop=True)
        v2_counts = like_quarterly.loc[
            like_quarterly["source_id"].eq(f"v2_main__{model_id}")
            & like_quarterly["top_risk_percentage"].eq(percentage),
            ["report_date", "active_banks_quarterly_denominator", "flagged_bank_quarters_in_quarter"],
        ].sort_values("report_date").reset_index(drop=True)
        group_size_audit_rows.append({
            "model_id": model_id, "top_risk_percentage": percentage,
            "quarters_compared": len(v1_counts),
            "active_denominators_identical": v1_counts[[
                "report_date", "active_banks_quarterly_denominator"
            ]].equals(v2_counts[["report_date", "active_banks_quarterly_denominator"]]),
            "flagged_group_sizes_identical": v1_counts[[
                "report_date", "flagged_bank_quarters_in_quarter"
            ]].equals(v2_counts[["report_date", "flagged_bank_quarters_in_quarter"]]),
        })
like_for_like_group_size_audit = pd.DataFrame(group_size_audit_rows)

comparison_discrimination.to_csv(
    NB17_OUTPUT_DIR / "v1_v2_final_test_discrimination_comparison.csv", index=False
)
comparison_capacity_with_ci.to_csv(
    NB17_OUTPUT_DIR / "v1_v2_final_test_capacity_comparison.csv", index=False
)
comparison_quarterly_capacity.to_csv(
    NB17_OUTPUT_DIR / "v1_v2_final_test_quarterly_capacity.csv", index=False
)
eligibility_losses.to_csv(NB17_OUTPUT_DIR / "v1_rows_lost_under_v2_eligibility.csv", index=False)
cert_key_audit.to_csv(NB17_OUTPUT_DIR / "v1_v2_common_key_cert_audit.csv", index=False)
like_for_like_group_size_audit.to_csv(
    NB17_OUTPUT_DIR / "v1_v2_like_for_like_group_size_audit.csv", index=False
)
pd.concat([
    v1_full_discrimination_replicates, like_discrimination_replicates
], ignore_index=True).to_parquet(
    NB17_OUTPUT_DIR / "v1_v2_discrimination_cluster_bootstrap_replicates.parquet",
    index=False, compression="snappy",
)
pd.concat([
    v1_full_capacity_replicates, like_capacity_replicates
], ignore_index=True).to_parquet(
    NB17_OUTPUT_DIR / "v1_v2_capacity_cluster_bootstrap_replicates.parquet",
    index=False, compression="snappy",
)

cell_07_checks = {
    "Comparison reports six models in each of three required sample definitions":
        len(comparison_discrimination) == 18
        and comparison_discrimination.groupby("sample_definition")["model_id"].nunique().eq(6).all(),
    "Archived Version 1 was never refitted":
        all(path.is_file() for path in V1_PREDICTION_PATHS.values()),
    "Version 1 restricted rows exactly cover all Version 2 eligible keys":
        all(len(frame) == len(v2_reference) for frame in v1_restricted_predictions.values()),
    "Row, positive and failed-bank eligibility losses are reported consistently":
        len(eligibility_losses) == 6 and eligibility_losses.drop(columns="model_id").nunique().max() == 1,
    "Version 1 full results retain their original non-like-for-like denominator":
        comparison_discrimination.loc[
            comparison_discrimination["sample_definition"].eq("full original Version 1 sample"),
            "like_for_like",
        ].eq(False).all(),
    "All like-for-like quarterly denominators and flagged group sizes are identical":
        like_for_like_group_size_audit[[
            "active_denominators_identical", "flagged_group_sizes_identical"
        ]].all().all(),
    "Every comparison uses the identical audited identity, bootstrap seed and replicate count":
        comparison_discrimination["bootstrap_identity"].eq("audited CERT component").all()
        and comparison_discrimination["bootstrap_seed"].eq(BOOTSTRAP_SEED).all()
        and comparison_discrimination["bootstrap_replicates"].eq(BOOTSTRAP_REPLICATES).all(),
    "Version 2 remains the main analysis and all differences are non-causal":
        comparison_discrimination.loc[
            comparison_discrimination["version"].eq("Version 2 main"), "analysis_priority"
        ].eq("main analysis").all()
        and not comparison_discrimination["causal_interpretation_permitted"].any(),
}
notebook_17_cell_07_gate = pd.DataFrame({
    "check": cell_07_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_07_checks.values()],
    "passed": cell_07_checks.values(),
})
notebook_17_cell_07_gate.to_csv(NB17_OUTPUT_DIR / "cell_07_v1_v2_comparison_gate.csv", index=False)
print("Notebook 17 Cell 7 validation gate:")
display(notebook_17_cell_07_gate)
print("\nVersion 1 rows lost under Version 2 eligibility:")
display(eligibility_losses.drop_duplicates(subset=eligibility_losses.columns.difference(["model_id"]).tolist()))
print("\nFinal-test discrimination comparison:")
display(comparison_discrimination)
failed = [name for name, passed in cell_07_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 7 blocked: " + "; ".join(failed))
print("\nCell 7 completed successfully. Version 1 remains secondary sensitivity evidence.")

In [ ]:
# Examine uninsured deposits descriptively on recent Version 2-eligible rows

if not notebook_17_cell_07_gate["passed"].all():
    raise AssertionError("Cell 8 blocked because Cell 7 did not pass.")

recent_labels = v2_reference[[
    "report_date", "cert", "idrssd", "bootstrap_identity", TARGET
]].copy()


def read_local_uninsured(path):
    path = Path(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    return pd.read_csv(path)


def fetch_fdic_uninsured():
    override = os.environ.get("NB17_UNINSURED_DEPOSIT_PATH", "").strip()
    if override:
        return read_local_uninsured(override), f"local override: {override}"
    headers = {"Accept": "application/json"}
    api_key = os.environ.get("FDIC_API_KEY", "").strip()
    if api_key:
        headers["X-API-Key"] = api_key
    fields = "CERT,REPDTE,DEP,DEPUNINS,ASSET,BKCLASS"
    first_date = recent_labels["report_date"].min().strftime("%Y%m%d")
    last_date = recent_labels["report_date"].max().strftime("%Y%m%d")
    filters = f"REPDTE:[{first_date} TO {last_date}]"
    records, offset, total = [], 0, None
    while total is None or offset < total:
        response = requests.get(
            FDIC_FINANCIALS_URL,
            params={
                "filters": filters, "fields": fields, "limit": 10000,
                "offset": offset, "sort_by": "REPDTE", "sort_order": "ASC",
                "format": "json",
            },
            headers=headers, timeout=90,
        )
        response.raise_for_status()
        payload = response.json()
        batch_raw = payload.get("data", [])
        batch = [item.get("data", item) if isinstance(item, dict) else item for item in batch_raw]
        records.extend(batch)
        meta = payload.get("meta", {}) if isinstance(payload, dict) else {}
        total = int(meta.get("total", payload.get("total", len(records))))
        if not batch:
            break
        offset += len(batch)
    return pd.DataFrame(records), "FDIC BankFind Suite financials API"


uninsured_available, uninsured_message = False, ""
try:
    raw_uninsured, uninsured_source = fetch_fdic_uninsured()
    raw_uninsured.rename(columns={column: column.upper() for column in raw_uninsured.columns}, inplace=True)
    required = {"CERT", "REPDTE", "DEP", "DEPUNINS"}
    if not required <= set(raw_uninsured.columns):
        raise KeyError("Missing FDIC fields: " + ", ".join(sorted(required - set(raw_uninsured.columns))))
    uninsured = raw_uninsured.copy()
    uninsured["cert"] = pd.to_numeric(uninsured["CERT"], errors="coerce")
    date_text = uninsured["REPDTE"].astype(str).str.replace(r"\.0$", "", regex=True)
    uninsured["report_date"] = pd.to_datetime(date_text, format="%Y%m%d", errors="coerce")
    uninsured["total_deposits_thousands"] = pd.to_numeric(uninsured["DEP"], errors="coerce")
    uninsured["estimated_uninsured_deposits_thousands"] = pd.to_numeric(
        uninsured["DEPUNINS"], errors="coerce"
    )
    if "BKCLASS" in uninsured:
        uninsured = uninsured.loc[
            uninsured["BKCLASS"].isin(["N", "NM", "SM"]) | uninsured["BKCLASS"].isna()
        ]
    uninsured = uninsured.loc[
        uninsured["cert"].notna() & uninsured["report_date"].notna()
    ].copy()
    uninsured["cert"] = uninsured["cert"].astype("int64")
    uninsured.drop_duplicates(["cert", "report_date"], keep="last", inplace=True)
    uninsured_recent = recent_labels.merge(
        uninsured[[
            "cert", "report_date", "total_deposits_thousands",
            "estimated_uninsured_deposits_thousands",
        ]],
        on=["cert", "report_date"], how="left", validate="one_to_one",
    )
    uninsured_recent["uninsured_deposit_share_raw"] = (
        uninsured_recent["estimated_uninsured_deposits_thousands"]
        / uninsured_recent["total_deposits_thousands"]
    )
    uninsured_recent["valid_uninsured_share"] = (
        uninsured_recent["total_deposits_thousands"].gt(0)
        & uninsured_recent["uninsured_deposit_share_raw"].between(0, 1)
    )
    uninsured_recent["uninsured_deposit_share"] = uninsured_recent[
        "uninsured_deposit_share_raw"
    ].where(uninsured_recent["valid_uninsured_share"])
    uninsured_available = bool(uninsured_recent["valid_uninsured_share"].any())
    uninsured_message = (
        "Downloaded and merged successfully"
        if uninsured_available else "Source returned no valid DEPUNINS/DEP observations"
    )
except Exception as exc:
    warnings.warn(f"Uninsured-deposit source unavailable: {exc}")
    uninsured_source = "unavailable"
    uninsured_message = f"{type(exc).__name__}: {exc}"
    uninsured_recent = recent_labels.copy()
    uninsured_recent["uninsured_deposit_share"] = np.nan
    uninsured_recent["valid_uninsured_share"] = False

coverage_rows = []
for report_date, group in uninsured_recent.groupby("report_date", sort=True):
    coverage_rows.append({
        "report_date": report_date,
        "v2_eligible_bank_quarters": len(group),
        "valid_uninsured_observations": int(group["valid_uninsured_share"].sum()),
        "coverage_rate": float(group["valid_uninsured_share"].mean()),
        "positive_bank_quarters": int(group[TARGET].sum()),
    })
uninsured_coverage = pd.DataFrame(coverage_rows)

distribution_rows = []
for target_value, group in uninsured_recent.groupby(TARGET):
    values = group.loc[group["valid_uninsured_share"], "uninsured_deposit_share"]
    distribution_rows.append({
        "group": "positive_bank_quarter" if target_value == 1 else "negative_bank_quarter",
        "eligible_observations": len(group), "valid_observations": len(values),
        "coverage_rate": len(values) / len(group),
        "mean_uninsured_share": values.mean(), "median_uninsured_share": values.median(),
        "q25_uninsured_share": values.quantile(0.25),
        "q75_uninsured_share": values.quantile(0.75),
        "descriptive_only": True,
    })
uninsured_distribution = pd.DataFrame(distribution_rows)
failed_latest = uninsured_recent.loc[uninsured_recent[TARGET].eq(1)].sort_values(
    ["bootstrap_identity", "report_date"]
).groupby("bootstrap_identity", as_index=False).tail(1)
failed_bank_uninsured = failed_latest[[
    "bootstrap_identity", "cert", "idrssd", "report_date", TARGET,
    "uninsured_deposit_share", "valid_uninsured_share",
]].sort_values("bootstrap_identity")

source_status = pd.DataFrame([{
    "available": uninsured_available, "source": uninsured_source,
    "message": uninsured_message, "field_numerator": "DEPUNINS",
    "field_denominator": "DEP",
    "analysis_role": "supplementary descriptive only; excluded from every model",
    "coverage_caveat": "reporting availability varies by bank size and period; compare coverage alongside distributions",
}])
source_status.to_csv(NB17_OUTPUT_DIR / "uninsured_deposit_source_status.csv", index=False)
uninsured_coverage.to_csv(NB17_OUTPUT_DIR / "uninsured_deposit_quarterly_coverage.csv", index=False)
uninsured_distribution.to_csv(NB17_OUTPUT_DIR / "uninsured_deposit_descriptive_summary.csv", index=False)
failed_bank_uninsured.to_csv(NB17_OUTPUT_DIR / "recent_failed_bank_uninsured_deposits.csv", index=False)
cell_08_checks = {
    "Recent uninsured analysis uses only locked Version 2 eligible keys and labels":
        len(uninsured_recent) == nb16_handoff["test_observations"]
        and int(uninsured_recent[TARGET].sum()) == nb16_handoff["positive_bank_quarters"],
    "The descriptive measure is exactly DEPUNINS divided by DEP":
        source_status["field_numerator"].iat[0] == "DEPUNINS"
        and source_status["field_denominator"].iat[0] == "DEP",
    "Valid shares are bounded without clipping invalid values":
        uninsured_recent.loc[
            uninsured_recent["valid_uninsured_share"], "uninsured_deposit_share"
        ].between(0, 1).all(),
    "Every recent Version 2 quarter has explicit coverage reporting":
        len(uninsured_coverage) == recent_labels["report_date"].nunique(),
    "Every recent failed-bank identity remains in the descriptive file":
        failed_bank_uninsured["bootstrap_identity"].nunique() == nb16_handoff["distinct_failed_banks"],
    "Uninsured deposits never enter a model feature set":
        "DEPUNINS" not in all_predictors and "uninsured_deposit_share" not in all_predictors,
    "Availability or exact unavailability is documented":
        source_status["message"].str.len().gt(0).all(),
}
notebook_17_cell_08_gate = pd.DataFrame({
    "check": cell_08_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_08_checks.values()],
    "passed": cell_08_checks.values(),
})
notebook_17_cell_08_gate.to_csv(NB17_OUTPUT_DIR / "cell_08_uninsured_deposit_gate.csv", index=False)
print("Notebook 17 Cell 8 validation gate:")
display(notebook_17_cell_08_gate)
print("\nUninsured-deposit source status:")
display(source_status)
print("\nDescriptive summary with coverage:")
display(uninsured_distribution)
failed = [name for name, passed in cell_08_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 8 blocked: " + "; ".join(failed))
print("\nCell 8 completed successfully. Uninsured deposits remain supplementary only.")

In [ ]:
# Produce interpretation registers, dissertation tables and final figures

if not notebook_17_cell_08_gate["passed"].all():
    raise AssertionError("Cell 9 blocked because Cell 8 did not pass.")

limitations = pd.DataFrame([
    ("Crisis status", "The 2008Q1-2012Q4 exercise is exploratory robustness evidence, not a second independent test."),
    ("Sparse pre-crisis failures", f"The eligible 2001Q2-2006Q4 training sample contains {training_positives:,} positive bank-quarters and {training_failed_banks:,} distinct failed-bank identities."),
    ("Predictor-selection overlap", "Version 2 predictor-selection data overlap the crisis evaluation period, so feature selection is not independent of the crisis outcomes."),
    ("Fixed crisis settings", "Crisis hyperparameters were fixed in Notebook 17 and not imported from Notebook 15; this avoids tuning on validation periods that overlap the crisis evaluation."),
    ("Version 1 full sample", "Archived Version 1 full-sample results use their original quarterly denominator and are non-like-for-like secondary evidence."),
    ("Version 1 crisis archive", "Archived unlagged Version 1 crisis tables are retained unchanged as non-like-for-like secondary sensitivity evidence and are not refitted in Notebook 17."),
    ("Like-for-like scope", "The like-for-like diagnostic holds the Version 2-eligible quarterly denominator and flagged group sizes fixed, but it does not create a causal experiment."),
    ("Timing and horizon", "Version 1-Version 2 differences reflect both predictor timing and the longer effective horizon from source_report_date; they are not a pure timing effect."),
    ("Feature sets", feature_set_summary["disclosure"].str.cat(sep="; ")),
    ("Version 2 priority", "Version 2 remains the main analysis regardless of whether its PR-AUC is higher or lower than archived Version 1."),
    ("Class-weighted scores", "Ranking is primary; Brier score and log loss are secondary diagnostics and the class-weighted scores are not assumed calibrated probabilities."),
    ("Capacity denominators", "Crisis flagged counts are reported both as totals across 20 evaluation quarters and as mean flagged banks per quarter; distinct-failed-bank capture is primary."),
    ("Uninsured deposits", "DEPUNINS/DEP is descriptive only and is interpreted together with explicit reporting coverage because availability varies structurally."),
], columns=["limitation", "required_statement"])

top_features = crisis_feature_effects.copy()
top_features["absolute_value"] = top_features["value"].abs()
top_features["within_model_rank"] = top_features.groupby("model_id")["absolute_value"].rank(
    method="first", ascending=False
)
top_features = top_features.loc[top_features["within_model_rank"].le(10)].sort_values(
    ["model_id", "within_model_rank"]
)
top_features["interpretation_boundary"] = np.where(
    top_features["effect_type"].eq("standardised_log_odds_coefficient"),
    "Conditional predictive association after training-only preprocessing; not causal",
    "Relative model importance; direction and causality are not identified",
)

def normalise_final_feature_effect_schema(frame):
    aliases = {
        "predictor": "feature",
        "effect_value": "value",
        "effect_measure": "effect_type",
    }
    normalised = frame.rename(columns={
        source: target
        for source, target in aliases.items()
        if source in frame.columns and target not in frame.columns
    }).copy()
    required = {"model_id", "feature", "value", "effect_type"}
    missing = sorted(required - set(normalised.columns))
    if missing:
        raise KeyError(
            "Notebook 16 feature-effect table cannot be normalised; missing columns: "
            + ", ".join(missing)
            + f". Available columns: {sorted(frame.columns)}"
        )
    normalised["value"] = pd.to_numeric(normalised["value"], errors="raise").astype("float64")
    normalised["effect_type"] = normalised["effect_type"].replace({
        "standardised_coefficient": "standardised_log_odds_coefficient",
        "impurity_importance": "gini_feature_importance",
        "gain_importance": "gain_feature_importance",
    })
    # The logistic intercept is not a predictor and is excluded from feature rankings.
    normalised = normalised.loc[normalised["feature"].ne("__intercept__")].copy()
    if not np.isfinite(normalised["value"]).all():
        raise AssertionError("Notebook 16 feature effects contain a non-finite value.")
    return normalised


final_v2_top_features = normalise_final_feature_effect_schema(final_v2_feature_effects)
final_v2_top_features["absolute_value"] = final_v2_top_features["value"].abs()
final_v2_top_features["within_model_rank"] = final_v2_top_features.groupby("model_id")[
    "absolute_value"
].rank(method="first", ascending=False)
final_v2_top_features = final_v2_top_features.loc[
    final_v2_top_features["within_model_rank"].le(10)
].sort_values(["model_id", "within_model_rank"])
final_v2_top_features["analysis_role"] = "Version 2 main interpretation"
final_v2_top_features["interpretation_boundary"] = np.where(
    final_v2_top_features["effect_type"].eq("standardised_log_odds_coefficient"),
    "Conditional predictive association after final-training preprocessing; not causal",
    "Relative final-model importance; direction and causality are not identified",
)
final_v2_calibration = final_v2_calibration.copy()
final_v2_calibration["diagnostic_priority"] = "secondary; no recalibration or reselection"
final_v2_horizon = final_v2_horizon.copy()
final_v2_horizon["required_interpretation"] = (
    "longer than Version 1 because predictors are measured at source_report_date=t-1 "
    "while the target remains (t,t+365]"
)
archived_v1_crisis_discrimination = archived_v1_crisis_discrimination.copy()
archived_v1_crisis_discrimination["version"] = "archived Version 1 unlagged"
archived_v1_crisis_discrimination["analysis_role"] = "secondary sensitivity evidence"
archived_v1_crisis_discrimination["like_for_like_with_v2_crisis"] = False
archived_v1_crisis_discrimination["refitted_in_notebook_17"] = False
archived_v1_crisis_capacity = archived_v1_crisis_capacity.copy()
archived_v1_crisis_capacity["version"] = "archived Version 1 unlagged"
archived_v1_crisis_capacity["analysis_role"] = "secondary sensitivity evidence"
archived_v1_crisis_capacity["like_for_like_with_v2_crisis"] = False
archived_v1_crisis_capacity["refitted_in_notebook_17"] = False

v2_like_disc = comparison_discrimination.loc[
    comparison_discrimination["version"].eq("Version 2 main")
].copy()
v1_like_disc = comparison_discrimination.loc[
    comparison_discrimination["source_id"].str.startswith("v1_restricted__")
].copy()
timing_sensitivity = v2_like_disc[["model_id", "roc_auc", "pr_auc"]].merge(
    v1_like_disc[["model_id", "roc_auc", "pr_auc"]],
    on="model_id", suffixes=("_v2", "_v1_restricted"), validate="one_to_one",
)
timing_sensitivity["roc_auc_difference_v2_minus_v1"] = (
    timing_sensitivity["roc_auc_v2"] - timing_sensitivity["roc_auc_v1_restricted"]
)
timing_sensitivity["pr_auc_difference_v2_minus_v1"] = (
    timing_sensitivity["pr_auc_v2"] - timing_sensitivity["pr_auc_v1_restricted"]
)
timing_sensitivity["interpretation"] = (
    "non-causal sensitivity: timing plus longer effective horizon; not a pure timing effect"
)
timing_sensitivity["version_2_remains_main"] = True

table_17_1 = crisis_sample_summary.copy()
table_17_2 = crisis_discrimination_with_ci.copy()
table_17_2["analysis_status"] = "exploratory robustness"
table_17_2["used_for_reselection"] = False
table_17_3 = crisis_capacity_with_ci.copy()
table_17_3["analysis_status"] = "exploratory robustness"
table_17_4 = comparison_discrimination.copy()
table_17_5 = comparison_capacity_with_ci.copy()
table_17_6 = eligibility_losses.merge(
    feature_set_summary.assign(join_key=1), how="cross"
)
table_17_7 = uninsured_distribution.copy()

table_17_1.to_csv(NB17_OUTPUT_DIR / "table_17_1_crisis_samples.csv", index=False)
table_17_2.to_csv(NB17_OUTPUT_DIR / "table_17_2_crisis_discrimination.csv", index=False)
table_17_3.to_csv(NB17_OUTPUT_DIR / "table_17_3_crisis_capacity.csv", index=False)
table_17_4.to_csv(NB17_OUTPUT_DIR / "table_17_4_v1_v2_discrimination.csv", index=False)
table_17_5.to_csv(NB17_OUTPUT_DIR / "table_17_5_v1_v2_capacity.csv", index=False)
eligibility_losses.to_csv(NB17_OUTPUT_DIR / "table_17_6_v1_eligibility_losses.csv", index=False)
feature_set_summary.to_csv(NB17_OUTPUT_DIR / "table_17_7_feature_set_disclosure.csv", index=False)
table_17_7.to_csv(NB17_OUTPUT_DIR / "table_17_8_uninsured_deposits.csv", index=False)
final_v2_top_features.to_csv(NB17_OUTPUT_DIR / "table_17_9_v2_main_interpretation.csv", index=False)
final_v2_calibration.to_csv(NB17_OUTPUT_DIR / "table_17_10_v2_secondary_calibration.csv", index=False)
final_v2_horizon.to_csv(NB17_OUTPUT_DIR / "table_17_11_v2_realised_horizon.csv", index=False)
archived_v1_crisis_discrimination.to_csv(
    NB17_OUTPUT_DIR / "table_17_12_archived_v1_crisis_discrimination.csv", index=False
)
archived_v1_crisis_capacity.to_csv(
    NB17_OUTPUT_DIR / "table_17_13_archived_v1_crisis_capacity.csv", index=False
)
top_features.to_csv(NB17_OUTPUT_DIR / "top_crisis_feature_interpretation.csv", index=False)
timing_sensitivity.to_csv(NB17_OUTPUT_DIR / "v1_v2_like_for_like_metric_differences.csv", index=False)
limitations.to_csv(NB17_OUTPUT_DIR / "dissertation_limitations_register.csv", index=False)

model_labels = {
    "logistic_regression__bank_only": "LR bank only",
    "logistic_regression__bank_plus_macro": "LR + macro",
    "random_forest__bank_only": "RF bank only",
    "random_forest__bank_plus_macro": "RF + macro",
    "xgboost__bank_only": "XGB bank only",
    "xgboost__bank_plus_macro": "XGB + macro",
    BASELINE_ID: "Low-equity baseline",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
plot_disc = crisis_discrimination_with_ci.copy()
x = np.arange(len(plot_disc))
axes[0].errorbar(
    x, plot_disc["pr_auc"],
    yerr=[
        np.maximum(0, plot_disc["pr_auc"] - plot_disc["pr_auc_ci_lower"]),
        np.maximum(0, plot_disc["pr_auc_ci_upper"] - plot_disc["pr_auc"]),
    ], fmt="o", capsize=3, color="#245D78",
)
axes[0].axhline(plot_disc["prevalence"].iloc[0], color="grey", linestyle="--", label="Prevalence")
axes[0].set_xticks(x, [model_labels.get(value, value) for value in plot_disc["source_id"]], rotation=45, ha="right")
axes[0].set_ylabel("PR-AUC")
axes[0].set_title("Exploratory crisis discrimination")
axes[0].legend()
for source_id, group in crisis_capacity_with_ci.groupby("source_id"):
    axes[1].plot(
        group["top_risk_percentage"] * 100,
        group["distinct_failed_bank_capture_rate"] * 100,
        marker="o", label=model_labels.get(source_id, source_id),
        linewidth=2 if source_id == leader_id else 1.2,
    )
axes[1].set_xlabel("Eligible active banks reviewed per quarter (%)")
axes[1].set_ylabel("Distinct failed banks captured (%)")
axes[1].set_title("Exploratory crisis primary capacity result")
axes[1].legend(fontsize=7)
fig.savefig(FIGURE_DIR / "figure_17_1_crisis_discrimination_and_capacity.png", dpi=220, bbox_inches="tight")
plt.show()

paired_plot = comparison_discrimination.loc[
    comparison_discrimination["scope"].eq("like_for_like_v2_denominator")
].copy()
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)
positions = np.arange(len(MODEL_IDS))
width = 0.36
v1_values = paired_plot.loc[paired_plot["version"].eq("Version 1 archived")].set_index("model_id").loc[MODEL_IDS]
v2_values = paired_plot.loc[paired_plot["version"].eq("Version 2 main")].set_index("model_id").loc[MODEL_IDS]
ax.bar(positions - width / 2, v1_values["pr_auc"], width, label="V1 restricted", color="#C27C2C")
ax.bar(positions + width / 2, v2_values["pr_auc"], width, label="V2 main", color="#2F6B5F")
ax.set_xticks(positions, [model_labels[value] for value in MODEL_IDS], rotation=35, ha="right")
ax.set_ylabel("PR-AUC")
ax.set_title("Like-for-like final-test sensitivity on Version 2-eligible keys")
ax.legend()
fig.savefig(FIGURE_DIR / "figure_17_2_v1_v2_like_for_like_pr_auc.png", dpi=220, bbox_inches="tight")
plt.show()

leader_capacity = comparison_capacity_with_ci.loc[
    comparison_capacity_with_ci["model_id"].eq(leader_id)
    & comparison_capacity_with_ci["scope"].eq("like_for_like_v2_denominator")
].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
for version, group in leader_capacity.groupby("version"):
    axes[0].plot(
        group["top_risk_percentage"] * 100,
        group["distinct_failed_bank_capture_rate"] * 100,
        marker="o", label=version,
    )
    axes[1].plot(
        group["top_risk_percentage"] * 100,
        group["positive_bank_quarter_capture_rate"] * 100,
        marker="o", label=version,
    )
axes[0].set(title="Primary: distinct failed-bank capture", xlabel="Banks reviewed per quarter (%)", ylabel="Capture (%)")
axes[1].set(title="Secondary: positive bank-quarter capture", xlabel="Banks reviewed per quarter (%)", ylabel="Capture (%)")
axes[0].legend(); axes[1].legend()
fig.savefig(FIGURE_DIR / "figure_17_3_leader_like_for_like_capacity.png", dpi=220, bbox_inches="tight")
plt.show()

leader_features = final_v2_top_features.loc[
    final_v2_top_features["model_id"].eq(leader_id)
].sort_values("value")
fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
colors = np.where(leader_features["value"].ge(0), "#2F6B5F", "#A04B43")
ax.barh(leader_features["feature"], leader_features["value"], color=colors)
ax.set_xlabel("Coefficient or normalised importance")
ax.set_title("Version 2 main: validation-selected leader interpretation")
fig.savefig(FIGURE_DIR / "figure_17_4_v2_main_leader_interpretation.png", dpi=220, bbox_inches="tight")
plt.show()

valid_uninsured = uninsured_recent.loc[uninsured_recent["valid_uninsured_share"]]
uninsured_groups = [
    valid_uninsured.loc[
        valid_uninsured[TARGET].eq(value), "uninsured_deposit_share"
    ].dropna()
    for value in [0, 1]
]
uninsured_figure_ready = uninsured_available and all(len(group) > 0 for group in uninsured_groups)
uninsured_figure_path = FIGURE_DIR / "figure_17_5_recent_uninsured_deposits.png"
if uninsured_figure_ready:
    fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
    ax.boxplot(
        uninsured_groups,
        tick_labels=["Negative bank-quarter", "Positive bank-quarter"],
        showfliers=False,
    )
    ax.set_ylabel("DEPUNINS / DEP")
    ax.set_title("Recent uninsured-deposit share (descriptive only)")
    fig.savefig(uninsured_figure_path, dpi=220, bbox_inches="tight")
    plt.show()
elif uninsured_figure_path.is_file():
    # Remove a stale optional figure from an earlier run if one target group has no coverage.
    uninsured_figure_path.unlink()

figure_manifest = pd.DataFrame([
    ("figures/figure_17_1_crisis_discrimination_and_capacity.png", "Exploratory crisis ranking and primary capacity", True),
    ("figures/figure_17_2_v1_v2_like_for_like_pr_auc.png", "Like-for-like V1-V2 PR-AUC sensitivity", True),
    ("figures/figure_17_3_leader_like_for_like_capacity.png", "Like-for-like leader capacity comparison", True),
    ("figures/figure_17_4_v2_main_leader_interpretation.png", "Version 2 main leader interpretation", True),
    ("figures/figure_17_5_recent_uninsured_deposits.png", "Recent uninsured-deposit description", False),
], columns=["figure_file", "purpose", "required"])
figure_manifest["exists"] = figure_manifest["figure_file"].map(
    lambda value: (NB17_OUTPUT_DIR / value).is_file()
)
figure_manifest.to_csv(NB17_OUTPUT_DIR / "cell_09_figure_manifest.csv", index=False)

reporting_statements = pd.DataFrame({"required_statement": [
    "Version 2 exact-t-1 lagged-bank results remain the main analysis regardless of whether their PR-AUC is higher or lower than archived Version 1.",
    "Archived Version 1 full-sample metrics retain the original quarterly denominator and are non-like-for-like secondary evidence.",
    "Archived unlagged Version 1 crisis results are retained unchanged as non-like-for-like secondary sensitivity evidence and are not refitted.",
    "The like-for-like diagnostic ranks Version 1 and Version 2 within the same quarterly denominator of Version 2-eligible active banks, yielding identical group sizes at 1%, 5%, 10% and 20%.",
    "Version 1-Version 2 differences are non-causal and reflect both predictor timing and the longer effective horizon from the predictor date; they are not a pure timing effect.",
    "The crisis-period exercise is exploratory, uses fixed settings independent of Notebook 15 tuning, and is limited by sparse pre-crisis failures and predictor-selection overlap.",
    "Crisis capacity reports flagged bank-quarters both as totals across 20 quarters and as mean flagged banks per evaluation quarter; distinct-failed-bank capture is primary and positive-bank-quarter capture secondary.",
    "Uninsured deposits are supplementary descriptive evidence only, with reporting coverage shown explicitly.",
]})
reporting_statements.to_csv(NB17_OUTPUT_DIR / "notebook_17_required_reporting_statements.csv", index=False)
cell_09_checks = {
    "All thirteen dissertation-facing tables exist":
        all((NB17_OUTPUT_DIR / name).is_file() for name in [
            "table_17_1_crisis_samples.csv", "table_17_2_crisis_discrimination.csv",
            "table_17_3_crisis_capacity.csv", "table_17_4_v1_v2_discrimination.csv",
            "table_17_5_v1_v2_capacity.csv", "table_17_6_v1_eligibility_losses.csv",
            "table_17_7_feature_set_disclosure.csv", "table_17_8_uninsured_deposits.csv",
            "table_17_9_v2_main_interpretation.csv", "table_17_10_v2_secondary_calibration.csv",
            "table_17_11_v2_realised_horizon.csv",
            "table_17_12_archived_v1_crisis_discrimination.csv",
            "table_17_13_archived_v1_crisis_capacity.csv",
        ]),
    "All four required figures exist": figure_manifest.loc[figure_manifest["required"], "exists"].all(),
    "The optional uninsured figure exists exactly when both target groups have valid data":
        bool(figure_manifest.loc[
            figure_manifest["figure_file"].str.contains("uninsured"), "exists"
        ].iat[0]) == uninsured_figure_ready,
    "Crisis tables remain exploratory and non-selective":
        table_17_2["analysis_status"].eq("exploratory robustness").all()
        and not table_17_2["used_for_reselection"].any(),
    "Version 2 main interpretation preserves secondary calibration and the longer horizon":
        final_v2_top_features["analysis_role"].eq("Version 2 main interpretation").all()
        and final_v2_calibration["diagnostic_priority"].str.contains("secondary").all()
        and final_v2_horizon["required_interpretation"].str.contains("longer than Version 1").all(),
    "Archived Version 1 crisis results remain unrefitted non-like-for-like sensitivity evidence":
        archived_v1_crisis_discrimination["analysis_role"].eq("secondary sensitivity evidence").all()
        and not archived_v1_crisis_discrimination["like_for_like_with_v2_crisis"].any()
        and not archived_v1_crisis_capacity["refitted_in_notebook_17"].any(),
    "Every limitation has explicit dissertation-ready language":
        len(limitations) >= 12 and limitations["required_statement"].str.len().gt(20).all(),
    "Required reporting statements preserve Version 2 priority and the horizon caveat":
        reporting_statements["required_statement"].str.contains("Version 2.*main", case=False, regex=True).any()
        and reporting_statements["required_statement"].str.contains("not a pure timing effect", case=False).any(),
}
notebook_17_cell_09_gate = pd.DataFrame({
    "check": cell_09_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_09_checks.values()],
    "passed": cell_09_checks.values(),
})
notebook_17_cell_09_gate.to_csv(NB17_OUTPUT_DIR / "cell_09_final_outputs_gate.csv", index=False)
print("Notebook 17 Cell 9 validation gate:")
display(notebook_17_cell_09_gate)
print("\nFigure manifest:")
display(figure_manifest)
print("\nDissertation limitations register:")
with pd.option_context("display.max_colwidth", None):
    display(limitations)
failed = [name for name, passed in cell_09_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 9 blocked: " + "; ".join(failed))
print("\nCell 9 completed successfully.")

In [ ]:
# Re-read outputs, audit input immutability and close Phase 6

if not notebook_17_cell_09_gate["passed"].all():
    raise AssertionError("Cell 10 blocked because Cell 9 did not pass.")

all_previous_gates = [
    notebook_17_cell_01_gate, notebook_17_cell_02_gate, notebook_17_cell_03_gate,
    notebook_17_cell_04_gate, notebook_17_cell_05_gate, notebook_17_cell_06_gate,
    notebook_17_cell_07_gate, notebook_17_cell_08_gate, notebook_17_cell_09_gate,
]
previous_checks = sum(len(gate) for gate in all_previous_gates)
previous_passes = sum(int(gate["passed"].sum()) for gate in all_previous_gates)
input_hashes_after = {str(path): sha256_file(path) for path in required_inputs}
input_immutability = pd.DataFrame([
    {
        "input_file": path, "sha256_before": input_hashes_before[path],
        "sha256_after": input_hashes_after[path],
        "unchanged": input_hashes_before[path] == input_hashes_after[path],
    }
    for path in input_hashes_before
])
input_immutability.to_csv(NB17_OUTPUT_DIR / "cell_10_input_immutability_audit.csv", index=False)

principal_purposes = {
    "cell_01_phase_6_contract.csv": "Predeclared Version 2 robustness and comparison contract",
    "cell_01_fixed_crisis_settings.csv": "Fixed settings independent of Notebook 15 tuning",
    "cell_03_crisis_sample_summary.csv": "Recalculated eligible Version 2 crisis counts",
    "cell_04_crisis_fit_register.csv": "Six crisis refits with training-only preprocessing",
    "crisis_discrimination_with_clustered_ci.csv": "Exploratory crisis ranking metrics with clustered intervals",
    "crisis_capacity_with_clustered_ci.csv": "Exploratory crisis primary and secondary capacity results",
    "v1_v2_final_test_discrimination_comparison.csv": "V1 full, V1 restricted and V2 discrimination comparison",
    "v1_v2_final_test_capacity_comparison.csv": "Original and like-for-like quarterly capacity comparison",
    "v1_rows_lost_under_v2_eligibility.csv": "Rows, positives and failed banks lost under V2 eligibility",
    "v1_v2_like_for_like_group_size_audit.csv": "Identical quarterly denominator and group-size audit",
    "v1_v2_feature_set_summary.csv": "Feature-set difference disclosure",
    "uninsured_deposit_descriptive_summary.csv": "Recent eligible-bank uninsured-deposit description",
    "dissertation_limitations_register.csv": "Mandatory final limitations language",
    "notebook_17_required_reporting_statements.csv": "Mandatory cautious reporting statements",
    "cell_09_figure_manifest.csv": "Final figure manifest",
    "table_17_1_crisis_samples.csv": "Dissertation crisis sample table",
    "table_17_2_crisis_discrimination.csv": "Dissertation crisis discrimination table",
    "table_17_3_crisis_capacity.csv": "Dissertation crisis capacity table",
    "table_17_4_v1_v2_discrimination.csv": "Dissertation V1-V2 discrimination sensitivity table",
    "table_17_5_v1_v2_capacity.csv": "Dissertation V1-V2 capacity sensitivity table",
    "table_17_6_v1_eligibility_losses.csv": "Dissertation eligibility-loss table",
    "table_17_7_feature_set_disclosure.csv": "Dissertation feature-set disclosure table",
    "table_17_8_uninsured_deposits.csv": "Dissertation uninsured-deposit table",
    "table_17_9_v2_main_interpretation.csv": "Version 2 main model interpretation table",
    "table_17_10_v2_secondary_calibration.csv": "Version 2 secondary calibration diagnostics",
    "table_17_11_v2_realised_horizon.csv": "Version 2 realised predictor-date horizon",
    "table_17_12_archived_v1_crisis_discrimination.csv": "Archived unlagged V1 crisis discrimination sensitivity",
    "table_17_13_archived_v1_crisis_capacity.csv": "Archived unlagged V1 crisis capacity sensitivity",
}
principal_paths = [NB17_OUTPUT_DIR / name for name in principal_purposes]
missing_principal = [str(path) for path in principal_paths if not path.is_file()]
if missing_principal:
    raise FileNotFoundError("Missing principal Notebook 17 outputs: " + ", ".join(missing_principal))
output_manifest = pd.DataFrame([
    {
        "file": str(path.relative_to(NB17_OUTPUT_DIR)),
        "purpose": principal_purposes[path.name],
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in principal_paths
])

final_handoff = {
    "schema_version": "v2_notebook_17_handoff_1.0",
    "notebook": "17_Robustness_Interpretation_and_Final_Outputs.ipynb",
    "status": "PASS / PHASE 6 COMPLETE",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "analysis_version": ANALYSIS_VERSION,
    "validation_selected_leader": leader_id,
    "test_based_reselection_performed": False,
    "version_2_remains_main_analysis": True,
    "crisis": {
        "status": "exploratory robustness",
        "training_period": "2001Q2-2006Q4",
        "purge_period": "2007Q1-2007Q4 not loaded",
        "evaluation_period": "2008Q1-2012Q4",
        "training_observations": len(crisis_train_data),
        "training_positive_bank_quarters": training_positives,
        "training_distinct_failed_banks": training_failed_banks,
        "evaluation_observations": len(crisis_eval_data),
        "evaluation_positive_bank_quarters": eval_positives,
        "evaluation_distinct_failed_banks": eval_failed_banks,
        "settings_source": "fixed in Notebook 17; independent of Notebook 15 tuning",
    },
    "v1_v2_comparison": {
        "version_1_refitted": False,
        "three_samples_reported": [
            "Version 1 full original denominator (non-like-for-like)",
            "Version 1 restricted to Version 2 eligible keys",
            "Version 2 main on Version 2 eligible keys",
        ],
        "eligibility_losses": eligibility_losses.iloc[0].drop(labels="model_id").to_dict(),
        "feature_set_summary": feature_set_summary.to_dict(orient="records"),
        "interpretation": "non-causal; timing plus longer effective horizon; not a pure timing effect",
    },
    "archived_v1_crisis": {
        "refitted": False,
        "like_for_like_with_v2_crisis": False,
        "role": "secondary sensitivity evidence",
        "discrimination_table": "table_17_12_archived_v1_crisis_discrimination.csv",
        "capacity_table": "table_17_13_archived_v1_crisis_capacity.csv",
    },
    "bootstrap": {
        "identity": "audited CERT component", "seed": BOOTSTRAP_SEED,
        "replicates": BOOTSTRAP_REPLICATES, "confidence_level": BOOTSTRAP_CONFIDENCE_LEVEL,
        "procedure": "multinomial cluster multiplicity bootstrap with fixed observed ranking memberships",
    },
    "capacity": {
        "fractions": list(TOP_RISK_PERCENTAGES),
        "primary": "distinct failed-bank capture",
        "secondary": "positive bank-quarter capture",
        "crisis_flagged_counts": "total across evaluation and mean per evaluation quarter",
    },
    "uninsured_deposits": {
        "available": uninsured_available, "measure": "DEPUNINS / DEP",
        "role": "supplementary descriptive only",
    },
    "principal_outputs": principal_purposes,
}
strict_json_dump(final_handoff, NB17_OUTPUT_DIR / "v2_final_dissertation_handoff.json")

final_summary = pd.DataFrame({
    "measure": [
        "Analysis version", "Notebook 16 status", "Validation-selected leader",
        "Test-based reselection performed", "Crisis status", "Crisis training period",
        "Crisis training observations", "Crisis training positive bank-quarters",
        "Crisis training distinct failed banks", "Crisis purge", "Crisis evaluation period",
        "Crisis evaluation observations", "Crisis evaluation positive bank-quarters",
        "Crisis evaluation distinct failed banks", "Crisis fixed specifications",
        "Version 1 refitted", "Comparison sample definitions", "Version 2 remains main",
        "Bootstrap identity", "Bootstrap seed", "Bootstrap replicates",
        "Capacity fractions", "Primary capacity result", "Secondary capacity result",
        "Uninsured-deposit role", "Previous Notebook 17 checks passed", "Pipeline status",
    ],
    "value": [
        ANALYSIS_VERSION, "PASS / FINAL VERSION 2 TEST LOCKED", leader_id, False,
        "exploratory robustness", "2001Q2-2006Q4", len(crisis_train_data),
        training_positives, training_failed_banks, "2007Q1-2007Q4 not loaded",
        "2008Q1-2012Q4", len(crisis_eval_data), eval_positives, eval_failed_banks,
        6, False, "V1 full; V1 restricted; V2 main", True,
        "audited CERT component", BOOTSTRAP_SEED, BOOTSTRAP_REPLICATES,
        str(TOP_RISK_PERCENTAGES), "distinct-failed-bank capture",
        "positive-bank-quarter capture", "supplementary descriptive only",
        f"{previous_passes}/{previous_checks}", "PASS / PHASE 6 COMPLETE",
    ],
})

reloaded_crisis = pd.read_csv(NB17_OUTPUT_DIR / "crisis_discrimination_with_clustered_ci.csv")
reloaded_comparison = pd.read_csv(NB17_OUTPUT_DIR / "v1_v2_final_test_discrimination_comparison.csv")
reloaded_capacity = pd.read_csv(NB17_OUTPUT_DIR / "v1_v2_final_test_capacity_comparison.csv")
reloaded_group_audit = pd.read_csv(NB17_OUTPUT_DIR / "v1_v2_like_for_like_group_size_audit.csv")
cell_10_checks = {
    "All nine earlier Notebook 17 gates passed": previous_passes == previous_checks,
    "Every loaded input remained byte-for-byte unchanged": input_immutability["unchanged"].all(),
    "All principal outputs exist and are non-empty": len(output_manifest) == len(principal_purposes) and output_manifest["bytes"].gt(0).all(),
    "Crisis results re-read seven sources with clustered intervals":
        len(reloaded_crisis) == 7
        and reloaded_crisis[["roc_auc_ci_lower", "roc_auc_ci_upper", "pr_auc_ci_lower", "pr_auc_ci_upper"]].notna().all().all(),
    "Final comparison re-reads exactly 18 model-sample rows":
        len(reloaded_comparison) == 18 and reloaded_comparison["model_id"].nunique() == 6,
    "Comparison capacity re-reads 72 model-fraction rows":
        len(reloaded_capacity) == 6 * 3 * 4,
    "All like-for-like denominators and group sizes remain identical":
        passed_values(reloaded_group_audit["active_denominators_identical"]).all()
        and passed_values(reloaded_group_audit["flagged_group_sizes_identical"]).all(),
    "Version 2 remains main without test-based reselection":
        final_handoff["version_2_remains_main_analysis"] is True
        and final_handoff["test_based_reselection_performed"] is False,
    "The final handoff states the comparison is not a pure timing effect":
        "not a pure timing effect" in final_handoff["v1_v2_comparison"]["interpretation"],
}
notebook_17_cell_10_gate = pd.DataFrame({
    "check": cell_10_checks.keys(),
    "result": ["PASS" if v else "FAIL" for v in cell_10_checks.values()],
    "passed": cell_10_checks.values(),
})
final_summary.to_csv(NB17_OUTPUT_DIR / "cell_10_final_summary.csv", index=False)
output_manifest.to_csv(NB17_OUTPUT_DIR / "cell_10_output_manifest.csv", index=False)
notebook_17_cell_10_gate.to_csv(NB17_OUTPUT_DIR / "cell_10_final_validation_gate.csv", index=False)
failed = [name for name, passed in cell_10_checks.items() if not passed]
if failed:
    raise AssertionError("Cell 10 blocked: " + "; ".join(failed))
print("Notebook 17 Cell 10 final validation gate:")
display(notebook_17_cell_10_gate)
print("\nNotebook 17 final summary:")
display(final_summary)
print("\nPrincipal output manifest:")
display(output_manifest)
print("\nNotebook 17 is PASS / PHASE 6 COMPLETE.")
print("Final dissertation handoff:", NB17_OUTPUT_DIR / "v2_final_dissertation_handoff.json")